In [ ]:
# CANDIDATE BUILD: strategy & operations profile (Warwick MBA, retail banking,
# unit head of 71) with INFRASTRUCTURE STRATEGY added - economic infrastructure:
# energy, water, transport, utilities, advisory and economic regulation.
#  MI/BI reporting, business analysis; analyst to specialist level, UK/London).
# OPENROUTER MANUAL REVIEW NOTE: the daily export is deterministic; the optional AI
# review uses OpenRouter via OPENROUTER_API_KEY and fills AI Remarks for a shortlist only.
# EXCEL NOTE: the Jobs sheet stores the FULL job description (never trimmed) and the
# Apply Link column is a real clickable hyperlink.
# CLEAN VERSION NOTE: Run all cells in Google Colab. Only the final export cell downloads Excel, with Jobs + Networking Tracker tabs.
!pip install requests feedparser pandas openpyxl beautifulsoup4 -q
print("âœ… Packages ready.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# HEMANTH DASU - ROLE DISCOVERY WORKFLOW
# Built from the Profile Bank (18 Sep 2026), not from guesswork.
#
# WHO THIS IS FOR
#   Mechanical engineer (B.Tech, KITS) who moved into Indian retail banking:
#   Virtual RM at Axis via Teleperformance, HNW portfolio management at HDFC,
#   Team Leader, then Unit Head at Axis Bank at 27 - 71 people through four
#   Senior Team Leads, seven product lines, commercial operations and
#   acquisition. Now completing a Full-Time MBA at Warwick Business School.
#   Independent research (CERF, IERF, the Conversion Gap paper) and two
#   self-built AI platforms. Entering the UK market for the first time.
#
# THE THROUGHLINE THE SCORING IS BUILT AROUND
#   Diagnosing a structural problem others had misread, building the fix
#   personally, securing buy-in from stakeholders who did not agree, and
#   proving it with a number.
#
# THREE THINGS THAT CHANGE HOW ROLES ARE SCORED HERE
#   1. VISA. The Graduate visa route gives two years of open work rights after
#      the MBA, so "we do not sponsor" postings stay viable at the point of
#      hiring. Sponsorship only becomes a real constraint at the two-year mark.
#      This notebook therefore does NOT penalise no-sponsorship wording.
#   2. INFRASTRUCTURE IS NOT A COLD PIVOT. CERF and IERF, the NGET Conversion
#      Gap work, the energy and utilities certifications, a Cadent final round
#      and a National Grid contact are real, citable engagement. Infrastructure
#      roles are scored as a sector ENTRY backed by published work, not as a
#      blind jump.
#   3. ACCESS, NOT SKILLS. Volume applications die at the automated filter;
#      referrals and published work produce interviews. Roles at employers where
#      a contact or a citable paper exists are flagged and ranked up.
# ─────────────────────────────────────────────────────────────────────────────

import os


# Colab Secrets and environment variables take precedence over anything stored here.
def _secret(name, default=""):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)


# ── CREDENTIALS ──────────────────────────────────────────────────────────────
# Two ways to supply the keys. Either works; Colab Secrets wins if both are set.
#   A) Paste them inline between the quotes below. Simplest, and it travels with
#      the file - so keep the file private.
#   B) Leave the quotes empty and add the same names under Colab Secrets.
# Only Adzuna and Reed need keys. Every other source in this notebook is keyless.
ADZUNA_APP_ID  = _secret("ADZUNA_APP_ID",  "")   # <-- paste your Adzuna app id here
ADZUNA_APP_KEY = _secret("ADZUNA_APP_KEY", "")   # <-- paste your Adzuna app key here
REED_API_KEY   = _secret("REED_API_KEY",   "")   # <-- paste your Reed API key here

# Email settings are read the same way. Nothing here sends email today; these are
# kept so any existing reference keeps resolving.
SENDER_EMAIL    = _secret("SENDER_EMAIL")
SENDER_PASSWORD = _secret("SENDER_PASSWORD")
RECEIVER_EMAIL  = _secret("RECEIVER_EMAIL")

_missing = [name for name, value in [
    ("ADZUNA_APP_ID", ADZUNA_APP_ID),
    ("ADZUNA_APP_KEY", ADZUNA_APP_KEY),
    ("REED_API_KEY", REED_API_KEY),
] if not value]
if _missing:
    print("!! Missing credentials: " + ", ".join(_missing))
    print("   Paste them into the marked lines above, or add them under Colab")
    print("   Secrets, then re-run this cell. Adzuna and Reed return nothing")
    print("   until they are set; every other source works without a key.")

# ── SEARCH SETTINGS ──────────────────────────────────────────────────────────
LOCATION           = "London"
SALARY_MIN         = 35000
RESULTS_PER_SOURCE = 25

CANDIDATE_SLUG = "hemanth"
OUTPUT_FILE    = f"{CANDIDATE_SLUG}_roles.xlsx"
AI_OUTPUT_FILE = f"{CANDIDATE_SLUG}_roles_ai_review.xlsx"

INCLUDE_INTERNSHIPS = False

# ── VISA POSITION (Profile Bank rule 21) ─────────────────────────────────────
# Graduate visa: two years of open work rights after the MBA. A "no sponsorship"
# advert is therefore NOT a blocker at the point of hiring.
GRADUATE_VISA_OPEN_WORK = True
VISA_NOTE = ("Graduate visa route gives two years of open work rights after the MBA, "
             "so no-sponsorship wording is not a blocker now; sponsorship matters "
             "only at the two-year mark.")

# ── TOOLS THAT MUST NEVER BE CLAIMED (Profile Bank rule 3) ───────────────────
# Used by the AI reviewer and printed on the workbook so no document ever drifts.
NEVER_CLAIM = [
    "SQL", "Salesforce", "Microsoft Dynamics", "any named CRM vendor",
    "Tableau", "QuickSight", "Advanced Power BI (working knowledge only)",
    "Scrum Master practical experience (certification held, no practice)",
]
SAFE_TO_CLAIM = [
    "Advanced Excel", "Power BI (working knowledge)", "Axis Bank internal CRM",
    "Python automation and REST API integration (self-built platforms)",
    "LLM-based evaluation workflows", "AgilePM v3 Foundation",
]

# ── ROLE TAXONOMY (from the Profile Bank honest role-fit map) ────────────────
# PRIMARY: a real, defensible shot. Matching evidence, no essential gates missing.
PRIMARY_TRACK_TERMS = {
    # "The closest match to seven years of actual lived experience."
    "Commercial Strategy & Operations": [
        "commercial strategy", "commercial operations", "commercial excellence",
        "commercial manager", "commercial lead", "business operations",
        "strategy and operations", "strategy & operations", "strategy operations",
        "strategic operations", "operating model", "operational excellence",
        "business performance", "performance manager", "revenue operations",
        "business manager", "general manager", "service operations",
    ],
    "Transformation & Operating Model": [
        "business transformation", "transformation manager", "transformation lead",
        "transformation analyst", "change manager", "change lead", "business change",
        "target operating model", "process redesign", "process improvement",
        "continuous improvement", "operational transformation",
        "banking transformation", "financial services transformation",
        "digital transformation", "fintech operations", "fintech strategy",
    ],
    # Expansion of an existing base, which is where every real sales number sits.
    "Business Development & Partnerships": [
        "partnerships manager", "strategic partnerships", "commercial partnerships",
        "partnership manager", "alliances", "channel manager", "account development",
        "client development", "relationship director", "portfolio growth",
        "business development manager", "market expansion", "customer success operations",
    ],
    # "Create structure from nothing" - SOPs, playbooks, KPI frameworks, clubs.
    "Builder / GTM & Enablement Operations": [
        "sales enablement", "gtm operations", "go-to-market operations",
        "revenue enablement", "commercial enablement", "playbook",
        "knowledge management", "business readiness", "capability building",
        "programme manager", "program manager", "pmo manager", "pmo lead",
        "portfolio manager", "delivery manager",
    ],
    # CERF and IERF make this a citable entry, not a cold pivot.
    "Infrastructure & Utilities Strategy": [
        "infrastructure strategy", "infrastructure advisory", "infrastructure planning",
        "asset strategy", "asset management strategy", "capital planning",
        "investment planning", "network strategy", "regulatory strategy",
        "economic regulation", "price control", "riio", "energy strategy",
        "transport strategy", "water strategy", "utilities strategy",
        "energy policy", "net zero strategy", "decarbonisation strategy",
        "stakeholder strategy", "customer strategy", "strategy and policy",
        "policy analyst", "regulation manager", "major projects",
    ],
    # "Understands AI capability and its practical limits", not an ML engineer.
    "AI-Adjacent Operations & Strategy": [
        "ai strategy", "ai operations", "ai adoption", "ai enablement",
        "ai transformation", "automation strategy", "intelligent automation",
        "decision intelligence", "ai product operations", "genai",
    ],
    # Bancassurance: 250% motor, 160% health, 120% life; personal lines only.
    "Insurance & Bancassurance Distribution": [
        "bancassurance", "insurance distribution", "insurance operations",
        "insurance partnerships", "protection", "life insurance", "health insurance",
        "motor insurance", "renewals", "insurance proposition", "distribution manager",
    ],
    "Customer Strategy & Experience": [
        "customer experience", "customer journey", "customer operations",
        "cx manager", "service design", "customer proposition", "proposition manager",
    ],
}

# SECONDARY: adjacent and plausible, scored a step below.
SECONDARY_TRACK_TERMS = {
    "Consulting (boutique preferred)": [
        "management consultant", "strategy consultant", "business consultant",
        "consulting manager", "advisory manager", "engagement manager",
        "transformation consultant", "operations consultant", "associate consultant",
    ],
    "Chief of Staff & Strategic Initiatives": [
        "chief of staff", "founder's office", "founders office", "ceo office",
        "office of the ceo", "strategic initiatives", "corporate strategy",
        "strategy manager", "business strategy",
    ],
    "Product & Proposition": [
        "product manager", "product operations", "product strategy",
        "platform manager", "digital platform manager", "digital manager",
    ],
    "Planning & Analytics": [
        "business planning", "workforce planning", "planning manager",
        "commercial analyst", "insight manager", "performance analyst",
        "business analyst", "fp&a", "financial planning and analysis",
    ],
}

# STRETCH: bridgeable, but the gap is real and must be named in the application.
STRETCH_TRACK_TERMS = {
    "New-business B2B sales": [
        "new business", "new logo", "business development representative",
        "sales development", "hunter", "pipeline generation",
    ],
    "Enterprise / public sector sales motion": [
        "enterprise sales", "government sales", "public sector sales",
        "bid manager", "capture manager", "pursuit lead",
    ],
    "Industrial & engineering strategy": [
        "industrial strategy", "manufacturing strategy", "engineering strategy",
        "technology commercialisation", "product commercialisation",
    ],
    "Sports & entertainment commercial": [
        "sports", "football club", "entertainment", "matchday", "fan engagement",
    ],
}

PRIMARY_TITLE_TERMS   = sorted({t for v in PRIMARY_TRACK_TERMS.values()   for t in v})
SECONDARY_TITLE_TERMS = sorted({t for v in SECONDARY_TRACK_TERMS.values() for t in v})
STRETCH_TITLE_TERMS   = sorted({t for v in STRETCH_TRACK_TERMS.values()   for t in v})
TIER1, TIER2, TIER3 = PRIMARY_TITLE_TERMS, SECONDARY_TITLE_TERMS, STRETCH_TITLE_TERMS

# ── DO NOT APPLY (Profile Bank rule 25: two or more unbridgeable essential gaps)
# These are not "score it low". They are "do not build a document for this".
DO_NOT_APPLY_RULES = {
    "Marketing function ownership": {
        "signals": ["marketing manager", "head of marketing", "brand manager",
                    "brand strategy", "marketing strategy", "campaign manager",
                    "digital marketing", "content marketing", "growth marketing",
                    "performance marketing", "seo", "paid media"],
        "reason": "No marketing-function ownership anywhere in the career history. "
                  "Campaign co-design with product and marketing teams is not the same thing.",
    },
    "Professional-services / legal BD": {
        "signals": ["legal business development", "law firm", "solicitors",
                    "professional services bd", "pitch and rfp", "rfp manager",
                    "bids and pitches", "partner development"],
        "reason": "Pitch/RFP and professional-services BD are hard essential gates "
                  "with zero bridgeable evidence. Dentons assessment put the honest "
                  "ceiling at 35-40 percent.",
    },
    "Quantitative / specialist finance": {
        "signals": ["cfa", "quantitative analyst", "quant researcher", "actuarial",
                    "investment analyst", "equity research", "credit analyst",
                    "portfolio manager investments", "trading"],
        "reason": "Deep finance specialisation was deliberately rejected as a direction.",
    },
    "ML / AI engineering": {
        "signals": ["machine learning engineer", "ml engineer", "data scientist",
                    "ai engineer", "research scientist", "deep learning",
                    "model development", "mlops"],
        "reason": "The AI projects show builder capability and product thinking, not "
                  "formal ML engineering. Deliberately rejected as a direction.",
    },
    "Entry-level IC banking reset": {
        "signals": ["relationship manager", "personal banker", "branch manager",
                    "customer service advisor", "telesales", "collections agent",
                    "mortgage adviser", "financial adviser"],
        "reason": "Restarting as a UK entry-level relationship manager was deliberately "
                  "rejected; it is a downward reset, not an explainable step.",
    },
}

# ── SEARCH KEYWORDS ──────────────────────────────────────────────────────────
CORE_SEARCH_KEYWORDS = [
    # Commercial strategy and operations - the closest match
    "commercial strategy manager", "commercial operations manager",
    "commercial excellence manager", "business operations manager",
    "strategy and operations manager", "strategy operations manager",
    "business performance manager", "operational excellence manager",
    "revenue operations manager", "operating model",
    # Transformation
    "business transformation manager", "transformation manager",
    "transformation analyst", "change manager", "business change manager",
    "banking transformation", "financial services transformation",
    "fintech operations", "fintech strategy", "digital transformation manager",
    # Business development and partnerships (existing-base expansion)
    "partnerships manager", "strategic partnerships manager",
    "commercial partnerships manager", "business development manager",
    "market expansion manager", "customer success operations",
    # Builder / enablement / programme
    "sales enablement manager", "gtm operations manager",
    "revenue enablement manager", "programme manager", "pmo manager",
    # Infrastructure and utilities - CERF/IERF backed
    "infrastructure strategy manager", "infrastructure advisory",
    "asset strategy manager", "capital planning manager",
    "investment planning manager", "economic regulation manager",
    "price control manager", "energy strategy manager",
    "utilities strategy manager", "transport strategy manager",
    "net zero strategy manager", "strategy and policy analyst",
    "regulation manager energy", "customer strategy energy",
    "stakeholder strategy manager",
    # AI-adjacent
    "ai strategy manager", "ai operations manager", "ai adoption lead",
    "automation strategy manager", "decision intelligence",
    # Insurance / bancassurance
    "bancassurance manager", "insurance distribution manager",
    "insurance operations manager", "insurance partnerships manager",
    # Customer and proposition
    "customer strategy manager", "customer experience manager",
    "proposition manager",
    # Consulting and chief of staff
    "management consultant", "strategy consultant", "transformation consultant",
    "chief of staff", "strategic initiatives manager",
]

INTERNSHIP_KEYWORDS = [
    "MBA internship", "summer associate", "strategy internship",
]
SEARCH_KEYWORDS = CORE_SEARCH_KEYWORDS + (INTERNSHIP_KEYWORDS if INCLUDE_INTERNSHIPS else [])

# Seniority: Unit Head of 71 plus an MBA. Manager to senior manager is the core.
SUPPORTED_SENIORITY = [
    "manager", "senior manager", "lead", "consultant", "senior consultant",
    "associate", "principal", "chief of staff", "specialist", "advisor", "adviser",
    "analyst",
]
UNSUPPORTED_SENIORITY = [
    "director", "managing director", "vice president", " vp ", "vp,",
    "chief executive", "chief operating", "chief commercial", "cfo", "coo",
    "partner", "global head", "group head",
]

RELEVANT_TITLES = [
    "strategy", "strategic", "commercial", "business", "operations", "operational",
    "transformation", "change", "operating model", "performance", "growth",
    "revenue", "partnerships", "business development", "programme", "program",
    "pmo", "portfolio", "enablement", "go-to-market", "gtm", "proposition",
    "customer", "consultant", "consulting", "advisory", "manager", "lead",
    "associate", "analyst", "policy", "regulation", "regulatory",
    "infrastructure", "asset", "capital", "investment", "energy", "water",
    "transport", "utilities", "net zero", "insurance", "bancassurance",
    "ai", "automation", "banking", "fintech",
]

# ── EXCLUSIONS ───────────────────────────────────────────────────────────────
HARD_EXCLUDE = [
    "software engineer", "software developer", "backend engineer", "frontend engineer",
    "full stack", "data engineer", "devops", "site reliability", "qa engineer",
    "test engineer", "network engineer", "security engineer", "ios developer",
    "android developer", "java developer", "python developer", "solutions architect",
    "accountant", "management accountant", "bookkeeper", "tax manager", "tax adviser",
    "solicitor", "lawyer", "legal counsel", "paralegal",
    "nurse", "nursing", "clinical", "care home", "pharmacist",
    "chef", "barista", "restaurant", "hotel operations", "housekeeping", "catering",
    "hospitality", "retail store", "store manager", "shop manager",
    "warehouse operations", "logistics operative", "driver", "hgv", "forklift",
    "electrician", "plumber", "installer", "field technician", "maintenance technician",
    "facilities manager", "estate manager", "cleaning operations",
    "teacher", "lecturer", "recruitment consultant",
    "graduate scheme", "apprentice", "apprenticeship",
    # Infrastructure delivery and site work, not strategy
    "site manager", "site engineer", "civil engineer", "structural engineer",
    "mechanical engineer", "electrical engineer", "design engineer",
    "quantity surveyor", "quantity surveying", "cad technician", "surveyor",
    "health and safety", "hse manager", "site supervisor", "foreman",
    "scaffolding", "plant operator", "commissioning engineer",
]

# SOFT: dropped only when nothing ties the advert back to this profile.
SOFT_EXCLUDE = [
    "audit manager", "internal audit", "compliance officer", "compliance manager",
    "underwriter", "claims handler", "payroll", "procurement manager",
    "supply chain", "hr manager", "people partner", "account manager",
    "sales manager", "sales executive",
]

RESCUE_TERMS = [
    "strategy", "strategic", "transformation", "operating model", "business operations",
    "commercial", "growth", "proposition", "chief of staff", "programme",
    "performance", "customer experience", "change management", "consulting",
    "infrastructure strategy", "asset strategy", "capital planning",
    "investment planning", "economic regulation", "price control", "enablement",
    "bancassurance", "insurance distribution",
]

EXCLUDE_TITLES = HARD_EXCLUDE + SOFT_EXCLUDE
if not INCLUDE_INTERNSHIPS:
    EXCLUDE_TITLES = EXCLUDE_TITLES + ["internship", "summer intern", "placement student"]

UNSUPPORTED_TECH_TERMS = [
    "spark", "scala", "hadoop", "kafka", "airflow", "kubernetes", "terraform",
    "microservices", "deep learning", "pytorch", "tensorflow", "c++", "golang",
    "cad", "revit", "autocad", "civil 3d", "finite element", "chartered engineer",
    # Tools the Profile Bank says must never be claimed.
    "advanced sql", "sql server", "t-sql", "tableau", "quicksight", "looker",
]

SOFTWARE_BA_TERMS = [
    "user stories", "acceptance criteria", "backlog grooming", "sdlc",
    "api specification", "technical specification", "system design", "wireframe",
]

# ── EVIDENCE ANGLES: where a citable asset or a real contact already exists ──
# Profile Bank section 8: this is an access problem, not a skills problem.
EVIDENCE_ANGLES = {
    "national grid": "IERF / Project Union work plus the NGET Conversion Gap paper; Amir is a dissertation interviewee who can refer internally.",
    "national gas": "IERF directly analyses Project Union, National Gas's hydrogen backbone plan.",
    "cadent": "Reached final stage before via Shivam Sharma's referral; RIIO allowed-revenue insight already developed.",
    "neso": "IERF cites NESO's own admission that the hydrogen blending economic case has not been specifically assessed.",
    "ofgem": "CERF/IERF gate framework and the RIIO revenue modelling insight.",
    "ofwat": "CERF/IERF regulatory-readiness framing transfers to price review work.",
    "barclays": "Matt Hammerstein, Barclays UK Corporate Bank CEO, is an ongoing conversation, not cold outreach.",
    "monzo": "Rupert Keeley call arranged; an independent Monzo strategy assignment already exists.",
    "warwick": "WBS Alumni Careers Ambassador; Sarah Jackson and Konstantina Dee are direct contacts.",
    "moasure": "MBA consulting project client - go-to-market strategy adopted into commercial planning.",
    "axis bank": "Seven years inside, including the Citibank India consumer integration.",
}

# ── LOCATION ─────────────────────────────────────────────────────────────────
NON_UK = [
    "united states", " usa", " us,", "u.s.", "canada", "australia", "japan",
    "singapore", "india", "germany", "france", "spain", "italy", "mexico", "brazil",
    "new york", "san francisco", "seattle", "boston", "los angeles", "washington",
    "chicago", "dallas", "miami", "austin", "tokyo", "sydney", "melbourne",
    "toronto", "paris", "berlin", "barcelona", "rome", "gurugram", "gurgaon",
    "mumbai", "bangalore", "bengaluru", "munich", "muenchen", "m?nchen",
    "frankfurt", "dublin", "ireland", "portugal", "lisbon", "sweden", "stockholm",
    "netherlands", "belgium", "poland", "hyderabad", "delhi", "sofia", "riga",
    "warsaw", "amsterdam", "madrid", "milan", "shanghai", "beijing", "hong kong",
    "seoul", "dubai", "abu dhabi", "riyadh", "apac", "emea", "latam",
    "remote - us", "remote us",
]

UK_LOCATION_SIGNALS = [
    "london", "united kingdom", "uk", "england", "remote uk", "remote - uk",
    "hybrid", "manchester", "edinburgh", "birmingham", "bristol", "leeds",
    "sheffield", "cardiff", "glasgow", "coventry", "warwick", "oxford",
    "cambridge", "belfast", "newcastle", "nottingham", "reading",
    "milton keynes", "swindon", "derby", "york", "ansty", "wales", "scotland",
    "northern ireland",
]

print("Profile loaded: Hemanth Dasu - commercial strategy and operations,")
print("transformation, BD and partnerships, builder/enablement, infrastructure")
print("and utilities, AI-adjacent, bancassurance, customer strategy.")
print(f"  Search keywords:     {len(SEARCH_KEYWORDS)}")
print(f"  Primary tracks:      {len(PRIMARY_TRACK_TERMS)} ({len(PRIMARY_TITLE_TERMS)} title terms)")
print(f"  Secondary / stretch: {len(SECONDARY_TRACK_TERMS)} / {len(STRETCH_TRACK_TERMS)}")
print(f"  Do-not-apply rules:  {len(DO_NOT_APPLY_RULES)}")
print(f"  Evidence angles:     {len(EVIDENCE_ANGLES)} employers with a citable asset or contact")
print(f"  Visa:                {VISA_NOTE[:60]}...")


In [ ]:
import re

LOCATION_MAP = {
    "london": ("UK", "Europe"), "united kingdom": ("UK", "Europe"), "england": ("UK", "Europe"),
    "remote uk": ("UK", "Europe"), "remote - uk": ("UK", "Europe"), "manchester": ("UK", "Europe"),
    "edinburgh": ("UK", "Europe"), "birmingham": ("UK", "Europe"), "bristol": ("UK", "Europe"),
    "leeds": ("UK", "Europe"), "sheffield": ("UK", "Europe"), "cardiff": ("UK", "Europe"),
    "glasgow": ("UK", "Europe"), "coventry": ("UK", "Europe"), "oxford": ("UK", "Europe"),
    "cambridge": ("UK", "Europe"),
    "amsterdam": ("Netherlands", "Europe"), "netherlands": ("Netherlands", "Europe"),
    "madrid": ("Spain", "Europe"), "spain": ("Spain", "Europe"), "barcelona": ("Spain", "Europe"),
    "paris": ("France", "Europe"), "france": ("France", "Europe"), "berlin": ("Germany", "Europe"),
    "germany": ("Germany", "Europe"), "munich": ("Germany", "Europe"), "muenchen": ("Germany", "Europe"),
    "frankfurt": ("Germany", "Europe"), "zurich": ("Switzerland", "Europe"), "switzerland": ("Switzerland", "Europe"),
    "dublin": ("Ireland", "Europe"), "ireland": ("Ireland", "Europe"), "milan": ("Italy", "Europe"),
    "rome": ("Italy", "Europe"), "italy": ("Italy", "Europe"), "lisbon": ("Portugal", "Europe"),
    "portugal": ("Portugal", "Europe"), "stockholm": ("Sweden", "Europe"), "sweden": ("Sweden", "Europe"),
    "warsaw": ("Poland", "Europe"), "poland": ("Poland", "Europe"), "sofia": ("Bulgaria", "Europe"),
    "riga": ("Latvia", "Europe"), "brussels": ("Belgium", "Europe"), "belgium": ("Belgium", "Europe"),
    "new york": ("USA", "North America"), "san francisco": ("USA", "North America"),
    "chicago": ("USA", "North America"), "seattle": ("USA", "North America"), "boston": ("USA", "North America"),
    "los angeles": ("USA", "North America"), "washington": ("USA", "North America"),
    "united states": ("USA", "North America"), "usa": ("USA", "North America"),
    "dallas": ("USA", "North America"), "miami": ("USA", "North America"), "austin": ("USA", "North America"),
    "toronto": ("Canada", "North America"), "canada": ("Canada", "North America"),
    "singapore": ("Singapore", "Asia Pacific"), "hong kong": ("Hong Kong", "Asia Pacific"),
    "tokyo": ("Japan", "Asia Pacific"), "japan": ("Japan", "Asia Pacific"),
    "sydney": ("Australia", "Asia Pacific"), "melbourne": ("Australia", "Asia Pacific"),
    "australia": ("Australia", "Asia Pacific"), "beijing": ("China", "Asia Pacific"),
    "shanghai": ("China", "Asia Pacific"), "china": ("China", "Asia Pacific"),
    "seoul": ("South Korea", "Asia Pacific"), "mumbai": ("India", "South Asia"),
    "delhi": ("India", "South Asia"), "gurugram": ("India", "South Asia"), "gurgaon": ("India", "South Asia"),
    "hyderabad": ("India", "South Asia"), "bangalore": ("India", "South Asia"), "bengaluru": ("India", "South Asia"),
    "india": ("India", "South Asia"), "dubai": ("UAE", "Middle East"), "abu dhabi": ("UAE", "Middle East"),
    "uae": ("UAE", "Middle East"), "riyadh": ("Saudi Arabia", "Middle East"), "saudi": ("Saudi Arabia", "Middle East"),
    "sao paulo": ("Brazil", "Latin America"), "brazil": ("Brazil", "Latin America"), "mexico": ("Mexico", "Latin America"),
}

NON_UK_COUNTRIES = {country for country, _ in LOCATION_MAP.values() if country != "UK"}


def normalise_text(value):
    return re.sub(r"\s+", " ", str(value or "").lower()).strip()


def get_country_continent(location):
    loc = normalise_text(location)
    if not loc or loc in {"see listing", "live now"}:
        return "Unknown", "Unknown"
    for keyword, result in LOCATION_MAP.items():
        if keyword in loc:
            return result
    if any(sig in loc for sig in UK_LOCATION_SIGNALS):
        return "UK", "Europe"
    return "Unknown", "Unknown"


def is_uk_loc(loc):
    loc_lower = normalise_text(loc)
    if not loc_lower:
        return True
    if any(non_uk in loc_lower for non_uk in NON_UK):
        return False
    country, _ = get_country_continent(loc_lower)
    if country in NON_UK_COUNTRIES:
        return False
    if country == "UK":
        return True
    # Unknown company-page locations are allowed into scoring, then reviewed by post-scoring filters.
    return True


def sponsorship_risk(text):
    t = normalise_text(text)
    no_sponsorship = [
        "no sponsorship", "cannot sponsor", "unable to sponsor", "will not sponsor",
        "does not sponsor", "do not sponsor", "must have right to work",
        "right to work in the uk", "eligible to work in the uk", "existing right to work",
        "without sponsorship", "sponsorship is not available",
    ]
    positive = ["visa sponsorship", "sponsorship available", "skilled worker", "sponsor licence", "graduate visa"]
    if any(phrase in t for phrase in no_sponsorship):
        return "High - right to work/no sponsorship wording"
    if any(phrase in t for phrase in positive):
        return "Low - sponsorship/visa friendly signal"
    return "Unknown"

# ── CANDIDATE-FIT VOCABULARY (from the Profile Bank) ─────────────────────────
# The throughline: diagnose a structural problem others misread, build the fix,
# secure buy-in from people who did not agree, prove it with a number.
FUNCTIONAL_TERMS = [
    "strategy", "strategic", "operating model", "business case", "transformation",
    "change management", "process improvement", "process redesign", "root cause",
    "continuous improvement", "operational excellence", "performance", "kpi",
    "commercial", "revenue", "growth", "p&l", "margin", "proposition",
    "customer experience", "customer journey", "segmentation", "go-to-market",
    "business development", "partnership", "forecasting", "demand", "capacity",
    "prioritisation", "roadmap", "business planning", "cost", "efficiency",
    "diagnostic", "analysis", "benchmarking", "stakeholder alignment",
]

LEADERSHIP_TERMS = [
    "lead a team", "team of", "line management", "people management",
    "manage a team", "direct reports", "coaching", "mentoring", "leadership",
    "develop talent", "capability", "span of control", "matrix",
]

# Lived regulatory experience is a genuine asset, not a stretch.
REGULATED_TERMS = [
    "regulated", "regulator", "regulatory", "compliance", "risk", "audit",
    "governance", "controls", "sla", "supplier governance", "policy",
    "fca", "pra", "ofgem", "ofwat", "ofcom", "conduct", "assurance",
]

DOMAIN_TERMS = [
    "bank", "banking", "retail banking", "financial services", "payments",
    "insurance", "bancassurance", "protection", "lending", "credit", "fintech",
    "infrastructure", "energy", "utilities", "water", "transport", "rail",
    "networks", "grid", "net zero", "consumer", "distribution",
]

# "Create structure from nothing" - SOPs, KPI frameworks, clubs, AI platforms.
BUILDER_TERMS = [
    "from scratch", "greenfield", "build out", "establish", "set up",
    "no precedent", "first hire", "playbook", "framework", "sop",
    "standard operating", "design and implement", "create the", "scale",
    "0 to 1", "zero to one", "define the", "shape the", "own the build",
]

TRANSFER_TERMS = [
    "stakeholder", "cross-functional", "delivery", "execution", "planning",
    "analysis", "analytical", "data-driven", "presentation", "communication",
    "problem solving", "commercial acumen", "business case", "reporting",
    "influencing", "workshops", "senior leadership", "negotiation",
]

SENIORITY_POINTS = [
    ("senior manager", 10), ("manager", 10), ("lead", 9), ("principal", 8),
    ("senior consultant", 9), ("consultant", 8), ("chief of staff", 10),
    ("senior analyst", 7), ("analyst", 6), ("associate", 7), ("specialist", 7),
    ("advisor", 7), ("adviser", 7), ("associate director", 5), ("head of", 4),
    ("director", 2), ("vice president", 2), ("managing director", 0), ("partner", 0),
]

# Gaps the Profile Bank says must be named plainly, once, rather than hidden.
HONEST_GAP_RULES = {
    "New-logo / cold-prospect hunting": [
        "new logo", "new business development", "cold calling", "cold outreach",
        "prospecting", "hunter", "pipeline generation", "self-generated pipeline",
    ],
    "Enterprise or public-sector sales motion": [
        "enterprise sales", "government sales", "public sector sales",
        "complex sales cycle", "six figure deals", "bid", "rfp", "tender",
    ],
    "Commercial-lines insurance product knowledge": [
        "commercial lines", "public liability", "professional indemnity",
        "sme risk", "commercial insurance", "underwriting",
    ],
    "Prior consulting-firm background": [
        "consulting background required", "previous consulting experience",
        "experience at a consultancy", "top-tier consulting", "mbb",
    ],
    "Sector employment history": [
        "experience in the energy sector", "utilities experience",
        "water industry experience", "rail industry experience",
        "infrastructure sector experience", "experience of price controls",
        "aviation experience", "sector expertise essential",
    ],
    "Named tooling not on the profile": [
        "advanced sql", "sql queries", "tableau", "quicksight", "looker",
        "salesforce", "dynamics 365", "power bi expert",
    ],
}


def term_hits(text, terms):
    return [term for term in terms if term in text]


def do_not_apply_reason(title, description=""):
    """
    Profile Bank rule 25: two or more unbridgeable essential gaps means the honest
    recommendation is not to apply at all. These roles are removed from the
    workbook rather than ranked low, and counted in the run summary.
    """
    t = normalise_text(title)
    combined = f"{t} {normalise_text(description)}"
    for rule_name, rule in DO_NOT_APPLY_RULES.items():
        if term_hits(t, rule["signals"]):
            return rule_name, rule["reason"]
        # Two or more independent signals in the body is enough on its own, even
        # when the title looks like a primary-track role: "Business Development
        # Manager" at a law firm is still law-firm BD. One passing mention is not,
        # so a commercial strategy role that references marketing once survives.
        if len(term_hits(combined, rule["signals"])) >= 2:
            return rule_name, rule["reason"]
    return "", ""


def exclusion_reason(title, description=""):
    t = normalise_text(title)
    combined = f"{t} {normalise_text(description)}"
    hard = term_hits(t, HARD_EXCLUDE)
    if hard:
        return f"EXCLUDED - unsuitable role type ({hard[0]})"
    rule_name, _ = do_not_apply_reason(title, description)
    if rule_name:
        return f"EXCLUDED - DO NOT APPLY: {rule_name}"
    soft = term_hits(t, SOFT_EXCLUDE)
    if soft and not term_hits(combined, RESCUE_TERMS):
        return f"EXCLUDED - {soft[0]} with no strategy/commercial content"
    return ""


def title_is_excluded(title, description=""):
    return bool(exclusion_reason(title, description))


GENERIC_TITLE_TERMS = {
    "manager", "business manager", "general manager", "operations manager",
    "commercial manager", "product manager", "delivery manager", "planning manager",
    "analyst", "associate", "consultant", "lead", "specialist", "business analyst",
}


def classify_track(title, description=""):
    """Returns (tier, track, points, matched_terms). Specific beats generic."""
    t = normalise_text(title)
    d = normalise_text(description)
    tiers = (
        ("Primary", PRIMARY_TRACK_TERMS, 32, 18),
        ("Secondary", SECONDARY_TRACK_TERMS, 22, 12),
        ("Stretch", STRETCH_TRACK_TERMS, 12, 6),
    )

    def best_match(haystack, group):
        candidates = []
        for track, terms in group.items():
            matched = [term for term in terms if term in haystack]
            if matched:
                specific = [term for term in matched if term not in GENERIC_TITLE_TERMS]
                rank = (1 if specific else 0, max(len(term) for term in (specific or matched)))
                candidates.append((rank, track, matched))
        if not candidates:
            return None
        _, track, matched = max(candidates, key=lambda x: x[0])
        return track, matched

    for tier, group, title_points, _ in tiers:
        found = best_match(t, group)
        if found:
            return tier, found[0], title_points, found[1]
    for tier, group, _, desc_points in tiers:
        found = best_match(d, group)
        if found:
            return f"{tier} (description only)", found[0], desc_points, found[1]
    return "Off-track", "Unmapped", 0, []


def evidence_angle(company, title="", description=""):
    """
    Where a citable paper or a real contact already exists. The Profile Bank is
    explicit that this is an access problem: referrals and published work are
    what produce interviews, so these roles are ranked up.
    """
    c = normalise_text(company)
    for key, angle in EVIDENCE_ANGLES.items():
        if key in c:
            return angle
    text = normalise_text(f"{title} {description}")
    if any(term in text for term in ["hydrogen", "gas network", "transmission",
                                     "energy networks", "riio", "price control"]):
        return ("IERF (Built Before Proven) and CERF give a citable, specific point of "
                "engagement with regulated network economics.")
    if any(term in text for term in ["ai adoption", "ai strategy", "human-ai",
                                     "decision support", "genai"]):
        return ("MBA dissertation on designing human-AI decision systems, plus two "
                "self-built AI platforms in daily use.")
    return ""


def honest_gaps(title, description=""):
    text = normalise_text(f"{title} {description}")
    return [name for name, signals in HONEST_GAP_RULES.items() if term_hits(text, signals)]


def extract_min_years(text):
    years = [int(y) for y in re.findall(r"(\d{1,2})\s*\+?\s*(?:years|yrs)", text)]
    years = [y for y in years if 0 < y <= 30]
    return min(years) if years else None


def seniority_points(title):
    t = normalise_text(title)
    matched = [(term, pts) for term, pts in SENIORITY_POINTS if term in t]
    if not matched:
        return 6, "unstated seniority"
    term, pts = min(matched, key=lambda x: x[1])
    return pts, term


def score_job(title, description="", location=""):
    """
    Ten dimensions to 100, measuring fit only. Honest by construction: gaps are
    named rather than absorbed, and roles with two or more unbridgeable essential
    gates are removed instead of being ranked low.

    Returns (total, track_points, reason, stretch, bucket) - unchanged shape.
    """
    t = normalise_text(title)
    d = normalise_text(description)
    text = f"{t} {d}"

    reason_excluded = exclusion_reason(title, description)
    if reason_excluded:
        return 0, 0, reason_excluded, 10, "D - Skip"
    if location and not is_uk_loc(location):
        return 0, 0, f"Non-UK: {location}", 10, "D - Skip"

    tier, track, track_score, track_terms = classify_track(title, description)

    functional = term_hits(text, FUNCTIONAL_TERMS)
    functional_score = min(len(functional) * 2, 15)

    leadership = term_hits(text, LEADERSHIP_TERMS)
    leadership_score = min(len(leadership) * 2.5, 10)

    regulated = term_hits(text, REGULATED_TERMS)
    regulated_score = min(len(regulated) * 2, 8)

    domain = term_hits(text, DOMAIN_TERMS)
    domain_score = min(len(domain) * 2, 10)

    builder = term_hits(text, BUILDER_TERMS)
    builder_score = min(len(builder) * 2.5, 8)

    supporting = len(functional) + len(leadership) + len(domain) + len(regulated)
    generic_note = ""
    title_is_generic = bool(track_terms) and all(term in GENERIC_TITLE_TERMS for term in track_terms)
    if track_score >= 22 and title_is_generic and supporting == 0:
        track_score = 12
        generic_note = " | generic title, no supporting content"

    seniority_score, seniority_term = seniority_points(title)
    if term_hits(t, UNSUPPORTED_SENIORITY):
        seniority_score = min(seniority_score, 3)

    min_years = extract_min_years(text)
    if min_years is None:
        experience_score, experience_note = 4, "no stated experience bar"
    elif min_years <= 8:
        experience_score, experience_note = 8, f"{min_years}+ yrs - within reach"
    elif min_years <= 10:
        experience_score, experience_note = 3, f"{min_years}+ yrs - slightly above"
    else:
        experience_score, experience_note = -8, f"{min_years}+ yrs - well above profile"
    if "mba" in text:
        experience_score = min(experience_score + 3, 8)
        experience_note += "; MBA named"

    transfer = term_hits(text, TRANSFER_TERMS)
    transfer_score = min(len(transfer) * 1.5, 8)

    location_score = 5 if (not location or any(sig in normalise_text(location)
                                               for sig in UK_LOCATION_SIGNALS)) else 2

    penalties = []
    unsupported_tech = term_hits(text, UNSUPPORTED_TECH_TERMS)
    if len(unsupported_tech) >= 2:
        penalties.append(("requires tooling not on the profile", -10))
    elif unsupported_tech:
        penalties.append((f"asks for {unsupported_tech[0]}", -4))

    software_ba = term_hits(text, SOFTWARE_BA_TERMS)
    if "business analyst" in t and len(software_ba) >= 2:
        penalties.append(("software-requirements BA", -10))

    gaps = honest_gaps(title, description)
    if gaps:
        penalties.append((f"named gap: {gaps[0]}", -5 if len(gaps) == 1 else -9))

    if tier == "Off-track":
        penalties.append(("no mapped track", -6))

    penalty_total = sum(points for _, points in penalties)

    total = (track_score + functional_score + leadership_score + regulated_score
             + domain_score + builder_score + seniority_score + experience_score
             + transfer_score + location_score + penalty_total)
    if tier == "Off-track":
        total = min(total, 35)
    total = int(max(0, min(round(total), 100)))

    stretch = calculate_stretch(title, description, "", total, track_score, transfer_score)
    bucket = make_bucket_inline(total, stretch)

    reason = (
        f"{tier}: {track}{generic_note} | track {int(track_score)}, function {functional_score:g}, "
        f"leadership {leadership_score:g}, regulated {regulated_score:g}, domain {domain_score:g}, "
        f"builder {builder_score:g}, seniority {seniority_score} ({seniority_term}), "
        f"experience {experience_score:+d} ({experience_note}), transferable {transfer_score:g}, "
        f"location {location_score}"
    )
    if penalties:
        reason += " | " + "; ".join(f"{label} {points}" for label, points in penalties)
    if gaps:
        reason += " | NAME THESE GAPS: " + "; ".join(gaps[:3])
    return total, int(track_score), reason, stretch, bucket


def make_bucket_inline(score, stretch):
    if score >= 70 and stretch <= 5:
        return "A - Apply Now"
    if score >= 60 and stretch <= 7:
        return "B - High Upside"
    if score >= 50 and stretch <= 9:
        return "C - Network First"
    return "D - Skip"


def calculate_stretch(title, description, company, score, interview_score=0, transfer_score=0):
    """
    1 = squarely within reach for a Unit Head of 71 with an MBA.
    10 = a seniority or an essential requirement the profile cannot reach.
    Note: no-sponsorship wording is NOT a stretch factor. The Graduate visa gives
    two years of open work rights after the MBA.
    """
    t = normalise_text(title)
    text = f"{t} {normalise_text(description)} {normalise_text(company)}"
    stretch = 3

    if term_hits(t, UNSUPPORTED_SENIORITY):
        stretch += 3
    elif "head of" in t:
        stretch += 2

    min_years = extract_min_years(text)
    if min_years is not None:
        if min_years >= 12:
            stretch += 3
        elif min_years >= 10:
            stretch += 1

    gap_count = len(honest_gaps(title, description))
    stretch += min(gap_count, 3)

    if len(term_hits(text, UNSUPPORTED_TECH_TERMS)) >= 2:
        stretch += 2

    # A citable paper or a warm contact is the difference between an application
    # that lands and one that dies at the filter.
    if evidence_angle(company, title, description):
        stretch = max(stretch - 1, 1)
    if "mba" in text:
        stretch = max(stretch - 1, 1)
    if any(term in t for term in SUPPORTED_SENIORITY):
        stretch = max(stretch - 1, 1)
    if any(term in t for term in PRIMARY_TITLE_TERMS):
        stretch = max(stretch - 1, 1)
    if interview_score >= 28 and transfer_score >= 5:
        stretch = max(stretch - 1, 1)
    if score >= 75:
        stretch = max(stretch - 1, 1)
    elif score < 50:
        stretch += 1

    return max(1, min(stretch, 10))


print("Scoring engine ready: 10 dimensions, do-not-apply gating, honest gaps")
print("named per role, evidence angles recognised, no visa penalty applied.")


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time

def fetch_adzuna(keyword):
    jobs = []
    url = (
        f"https://api.adzuna.com/v1/api/jobs/gb/search/1"
        f"?app_id={ADZUNA_APP_ID}&app_key={ADZUNA_APP_KEY}"
        f"&results_per_page={RESULTS_PER_SOURCE}"
        f"&what={requests.utils.quote(keyword)}"
        f"&where={requests.utils.quote(LOCATION)}"
        f"&salary_min={SALARY_MIN}&sort_by=date"
        f"&content-type=application/json"
    )
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        for item in r.json().get("results", []):
            jobs.append({
                "title":   item.get("title", ""),
                "company": item.get("company", {}).get("display_name", ""),
                "location": item.get("location", {}).get("display_name", ""),
                "salary":  f"{item.get('salary_min','')}â€“{item.get('salary_max','')}",
                "posted":  item.get("created", "")[:10],
                "desc":    item.get("description", ""),
                "url":     item.get("redirect_url", ""),
                "source":  "Adzuna",
            })
    except Exception as e:
        print(f"  Adzuna error [{keyword}]: {e}")
    return jobs

def fetch_reed(keyword):
    jobs = []
    url = (
        f"https://www.reed.co.uk/api/1.0/search"
        f"?keywords={requests.utils.quote(keyword)}"
        f"&locationName={requests.utils.quote(LOCATION)}"
        f"&minimumSalary={SALARY_MIN}"
        f"&resultsToTake={RESULTS_PER_SOURCE}"
    )
    try:
        r = requests.get(url, auth=(REED_API_KEY, ""), timeout=10)
        r.raise_for_status()
        for item in r.json().get("results", []):
            jobs.append({
                "title":   item.get("jobTitle", ""),
                "company": item.get("employerName", ""),
                "location": item.get("locationName", ""),
                "salary":  f"{item.get('minimumSalary','')}â€“{item.get('maximumSalary','')}",
                "posted":  item.get("date", "")[:10],
                "desc":    item.get("jobDescription", ""),
                "url":     item.get("jobUrl", ""),
                "source":  "Reed",
            })
    except Exception as e:
        print(f"  Reed error [{keyword}]: {e}")
    return jobs

print("Searching Adzuna + Reed...")
print("-" * 55)
board_jobs = []
for kw in SEARCH_KEYWORDS:
    a = fetch_adzuna(kw)
    r = fetch_reed(kw)
    board_jobs += a + r
    print(f"  {kw[:45]:<45} A:{len(a)} R:{len(r)}")
print("-" * 55)
print(f"âœ… Job boards: {len(board_jobs)}")

In [ ]:
import time
from bs4 import BeautifulSoup

headers_req = {"User-Agent": "Mozilla/5.0"}

# Cleans API-provided HTML descriptions while preserving plain text.
def clean_description(raw):
    if not raw:
        return ""
    return BeautifulSoup(raw, "html.parser").get_text(" ", strip=True)

company_jobs = []

GREENHOUSE = {
    # Fintech / challenger banks
    "Monzo":            "monzo",
    "GoCardless":       "gocardless",
    "Capital on Tap":   "capitalontap",
    "Adyen":            "adyen",
    "Stripe":           "stripe",
    "Marqeta":          "marqeta",
    "Dojo":             "dojo",
    "ComplyAdvantage":  "complyadvantage",
    "Tide":             "tide",
    "SumUp":            "sumup",
    "Liberis":          "liberis",
    "TrueLayer":        "truelayer",
    "Onfido":           "onfido",
    "Quantexa":         "quantexa",
    "Airbnb":           "airbnb",
    "Deliveroo":        "deliveroo",
    "Plaid":            "plaid",
    "Funding Circle":   "fundingcircle",
    "Freetrade":        "freetrade",
    "Marshmallow":      "marshmallow",
    "Iwoca":            "iwoca",
    "Nutmeg":           "nutmeg",
    "Zopa":             "zopabank",
    "Experian":         "experian",
    # Consulting
    "AlixPartners":     "alixpartners",
    "PA Consulting":    "paconsulting",
    "Capgemini":        "capgemini",
    "Slalom":           "slalom",
    "North Highland":   "northhighland",
    # Other
    "Octopus Energy GH": "octoenergy",
}

LEVER = {
    "Octopus Energy":   "octoenergy",
    "Moneybox":         "moneyboxapp",
    "OakNorth":         "oaknorth.ai",
    "Allica Bank":      "allica-bank",
    "Zego":             "zego",
    "TrueLayer LV":     "truelayer",
}

DIRECT = {
    "Deloitte":     "https://apply.deloitte.com/careers/SearchJobs/strategy",
    "EY Parthenon": "https://careers.ey.com/ey/search/?q=strategy+operations",
    "Bain":         "https://www.bain.com/careers/find-a-role/",
    "Barclays":     "https://search.jobs.barclays/search-jobs/London/22160/4",
    "HSBC":         "https://mycareer.hsbc.com/en_GB/external/SearchJobs",
    "Accenture":    "https://www.accenture.com/gb-en/careers/jobsearch",
    "BlackRock":    "https://careers.blackrock.com/",
    "Fidelity":     "https://careers.fidelityinternational.com/",
    "Schroders":    "https://www.schroders.com/en-gb/uk/individual/careers/",
}

NON_UK = [
    "united states", " us,", "canada", "australia", "japan",
    "singapore", "india", "germany", "france", "spain", "italy",
    "mexico", "brazil", "new york", "san francisco", "seattle",
    "tokyo", "sydney", "toronto", "paris", "berlin", "barcelona",
    "rome", "gurugram", "munich", "mÃ¼nchen", "dublin", "ireland",
    "portugal", "sweden", "netherlands", "belgium", "poland",
    "hyderabad", "delhi", "sofia", "riga", "warsaw", "amsterdam",
    "madrid", "milan", "shanghai", "beijing", "chicago",
]

def is_uk_loc(loc):
    if not loc: return True
    return not any(x in loc.lower() for x in NON_UK)

print("Checking company pages...")
print("-" * 55)

for company, token in GREENHOUSE.items():
    for base_url in [
        f"https://boards-api.greenhouse.io/v1/boards/{token}/jobs?content=true",
        f"https://boards.greenhouse.io/{token}",
    ]:
        try:
            r = requests.get(base_url, headers=headers_req, timeout=12)
            r.raise_for_status()
            found = 0
            if "api" in base_url:
                for job in r.json().get("jobs", []):
                    title = job.get("title", "")
                    offices = job.get("offices", [])
                    location = offices[0].get("name", "") if offices else ""
                    if not is_uk_loc(location): continue
                    # High recall: do not require title keyword match before AI review
                    if title_is_excluded(title): continue
                    company_jobs.append({
                        "title": title, "company": company,
                        "location": location or "UK",
                        "salary": "See listing", "posted": "Live now",
                        "desc": clean_description(job.get("content", "")) or title,
                        "description_available": "Yes" if job.get("content") else "No",
                        "description_quality": "Full API" if job.get("content") else "Title Only",
                        "source": "Career Page",
                        "url": job.get("absolute_url", ""),
                    })
                    found += 1
            else:
                soup = BeautifulSoup(r.text, "html.parser")
                for a in soup.find_all("a", href=True):
                    title = a.get_text(strip=True)
                    href = a["href"]
                    if len(title) < 8 or len(title) > 110: continue
                    # High recall: do not require title keyword match before AI review
                    if title_is_excluded(title): continue
                    full_url = href if href.startswith("http") else f"https://boards.greenhouse.io{href}"
                    company_jobs.append({
                        "title": title, "company": company,
                        "location": "UK", "salary": "See listing",
                        "posted": "Live now", "desc": title,
                        "source": "Career Page", "url": full_url,
                    })
                    found += 1
            print(f"  {company:<25} {found} roles")
            break
        except:
            continue
    time.sleep(0.3)

for company, slug in LEVER.items():
    url = f"https://api.lever.co/v0/postings/{slug}?mode=json"
    try:
        r = requests.get(url, headers=headers_req, timeout=12)
        r.raise_for_status()
        found = 0
        for job in r.json():
            title = job.get("text", "")
            location = job.get("categories", {}).get("location", "")
            if not is_uk_loc(location): continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            company_jobs.append({
                "title": title, "company": company,
                "location": location or "UK",
                "salary": "See listing", "posted": "Live now",
                "desc": clean_description(job.get("descriptionPlain") or job.get("description") or "") or title,
                "description_available": "Yes" if (job.get("descriptionPlain") or job.get("description")) else "No",
                "description_quality": "Full API" if (job.get("descriptionPlain") or job.get("description")) else "Title Only",
                "source": "Career Page",
                "url": job.get("hostedUrl", ""),
            })
            found += 1
        print(f"  {company:<25} {found} roles")
    except:
        print(f"  {company:<25} could not reach")
    time.sleep(0.3)

for company, url in DIRECT.items():
    try:
        r = requests.get(url, headers=headers_req, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")
        found = 0
        for a in soup.find_all("a", href=True):
            title = a.get_text(strip=True)
            href = a["href"]
            if len(title) < 8 or len(title) > 110: continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            full_url = href if href.startswith("http") else url.split("/")[0] + "//" + url.split("/")[2] + href
            company_jobs.append({
                "title": title, "company": company,
                "location": "London/UK", "salary": "See listing",
                "posted": "Live now", "desc": title,
                "source": "Career Page", "url": full_url,
            })
            found += 1
        print(f"  {company:<25} {found} roles")
    except:
        print(f"  {company:<25} could not reach")
    time.sleep(0.5)

# (Removed: hardcoded placeholder roles from the previous candidate's build.)

print("-" * 55)
print(f"âœ… Total company jobs: {len(company_jobs)}")


In [ ]:
# Cleans API-provided HTML descriptions while preserving plain text.
def clean_description(raw):
    if not raw:
        return ""
    return BeautifulSoup(raw, "html.parser").get_text(" ", strip=True)

# Add missing companies that were in the bigger list
GREENHOUSE_EXTRA = {
    # Data, regtech and financial-data employers added for this profile.
    "Quantexa DG":      "quantexa",
    "ComplyAdvantage DG": "complyadvantage",
    "Thought Machine":  "thoughtmachine",
    "Starling Bank":    "starlingbank",
    "FNZ":              "fnz",
    "Clearbank":        "clearbank",
    "Onfido":           "onfido",
    "Quantexa":         "quantexa",
    "Plaid":            "plaid",
    "Funding Circle":   "fundingcircle",
    "Freetrade":        "freetrade",
    "Marshmallow":      "marshmallow",
    "Iwoca":            "iwoca",
    "Nutmeg":           "nutmeg",
    "Experian":         "experian",
    "PA Consulting":    "paconsulting",
    "Capgemini":        "capgemini",
    "Slalom":           "slalom",
    "North Highland":   "northhighland",
    "Zopa":             "zopabank",
    "Checkout.com":     "checkoutdotcom",
    "Klarna":           "klarna",
    "Pleo":             "pleo",
    "Dojo Extra":       "dojo",
}

LEVER_EXTRA = {
    "Zego":             "zego",
    "Allica Bank":      "allica-bank",
    "Cleo":             "meetcleo",
    "Teneo":            "teneo",
}

print("Adding missing companies...")
print("-" * 55)
extra = 0

for company, token in GREENHOUSE_EXTRA.items():
    for base_url in [
        f"https://boards-api.greenhouse.io/v1/boards/{token}/jobs?content=true",
        f"https://boards.greenhouse.io/{token}",
    ]:
        try:
            r = requests.get(base_url, headers=headers_req, timeout=12)
            r.raise_for_status()
            found = 0
            if "api" in base_url:
                for job in r.json().get("jobs", []):
                    title = job.get("title", "")
                    offices = job.get("offices", [])
                    location = offices[0].get("name", "") if offices else ""
                    if not is_uk_loc(location): continue
                    # High recall: do not require title keyword match before AI review
                    if title_is_excluded(title): continue
                    company_jobs.append({
                        "title": title, "company": company,
                        "location": location or "UK",
                        "salary": "See listing", "posted": "Live now",
                        "desc": clean_description(job.get("content", "")) or title,
                        "description_available": "Yes" if job.get("content") else "No",
                        "description_quality": "Full API" if job.get("content") else "Title Only",
                        "source": "Career Page",
                        "url": job.get("absolute_url", ""),
                    })
                    found += 1
            else:
                soup = BeautifulSoup(r.text, "html.parser")
                for a in soup.find_all("a", href=True):
                    title = a.get_text(strip=True)
                    href = a["href"]
                    if len(title) < 8 or len(title) > 110: continue
                    # High recall: do not require title keyword match before AI review
                    if title_is_excluded(title): continue
                    full_url = href if href.startswith("http") else f"https://boards.greenhouse.io{href}"
                    company_jobs.append({
                        "title": title, "company": company,
                        "location": "UK", "salary": "See listing",
                        "posted": "Live now", "desc": title,
                        "source": "Career Page", "url": full_url,
                    })
                    found += 1
            print(f"  {company:<25} {found} roles")
            extra += found
            break
        except:
            continue
    time.sleep(0.3)

for company, slug in LEVER_EXTRA.items():
    url = f"https://api.lever.co/v0/postings/{slug}?mode=json"
    try:
        r = requests.get(url, headers=headers_req, timeout=12)
        r.raise_for_status()
        found = 0
        for job in r.json():
            title = job.get("text", "")
            location = job.get("categories", {}).get("location", "")
            if not is_uk_loc(location): continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            company_jobs.append({
                "title": title, "company": company,
                "location": location or "UK",
                "salary": "See listing", "posted": "Live now",
                "desc": clean_description(job.get("descriptionPlain") or job.get("description") or "") or title,
                "description_available": "Yes" if (job.get("descriptionPlain") or job.get("description")) else "No",
                "description_quality": "Full API" if (job.get("descriptionPlain") or job.get("description")) else "Title Only",
                "source": "Career Page",
                "url": job.get("hostedUrl", ""),
            })
            found += 1
        print(f"  {company:<25} {found} roles")
        extra += found
    except:
        print(f"  {company:<25} could not reach")
    time.sleep(0.3)

print("-" * 55)
print(f"Extra roles added:    {extra}")
print(f"Total company jobs:   {len(company_jobs)}")


In [ ]:
# Old duplicate all-in-one export cell disabled.
# The notebook now uses the main fetch cells plus the final export cell only.
print("Skipped old duplicate export cell. Final Excel download happens in the last scoring/export cell.")


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import files
import time
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime

headers_req = {"User-Agent": "Mozilla/5.0"}

# Cleans API-provided HTML descriptions while preserving plain text.
def clean_description(raw):
    if not raw:
        return ""
    return BeautifulSoup(raw, "html.parser").get_text(" ", strip=True)



def request_with_backoff(url, retries=3, base_sleep=1.0, **kwargs):
    for attempt in range(retries):
        try:
            response = requests.get(url, **kwargs)
            if response.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"Retryable status {response.status_code}")
            return response
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(base_sleep * (2 ** attempt))


# â”€â”€ Company pages â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Keep anything the earlier company cells already collected instead of
# discarding it; duplicates are removed by URL in the final export cell.
company_jobs = list(globals().get("company_jobs", []))

GREENHOUSE = {
    "Monzo":            "monzo",
    "GoCardless":       "gocardless",
    "Capital on Tap":   "capitalontap",
    "Adyen":            "adyen",
    "Stripe":           "stripe",
    "Marqeta":          "marqeta",
    "Dojo":             "dojo",
    "ComplyAdvantage":  "complyadvantage",
    "Tide":             "tide",
    "SumUp":            "sumup",
    "Liberis":          "liberis",
    "TrueLayer":        "truelayer",
    "Onfido":           "onfido",
    "Airbnb":           "airbnb",
    "AlixPartners":     "alixpartners",
    "Funding Circle":   "fundingcircle",
    "Freetrade":        "freetrade",
    "Marshmallow":      "marshmallow",
    "Iwoca":            "iwoca",
    "Zopa":             "zopabank",
    "Plaid":            "plaid",
    "Experian":         "experian",
    "Checkout.com":     "checkoutdotcom",
    "Klarna":           "klarna",
    "Pleo":             "pleo",
    "Capgemini":        "capgemini",
    "Slalom":           "slalom",
}

LEVER = {
    "Octopus Energy":   "octoenergy",
    "Moneybox":         "moneyboxapp",
    "OakNorth":         "oaknorth.ai",
    "Zego":             "zego",
    "Cleo":             "meetcleo",
    "Teneo":            "teneo",
}

DIRECT = {
    "National Grid":  "https://careers.nationalgrid.com/search/jobs",
    "SSE":            "https://www.sse.com/careers/search-and-apply/",
    "Network Rail":   "https://www.networkrail.co.uk/careers/search-jobs/",
    "Arup":           "https://www.arup.com/careers/job-search/",
    "Mott MacDonald": "https://www.mottmac.com/careers/job-search",
    "Baringa":        "https://www.baringa.com/en/careers/vacancies/",
    "LSEG":         "https://www.lseg.com/en/careers/open-roles",
    "Moody's":      "https://careers.moodys.com/jobs",
    "S&P Global":   "https://careers.spglobal.com/jobs",
    "Grant Thornton": "https://www.grantthornton.co.uk/careers/",
    "NatWest":      "https://jobs.natwestgroup.com/search-jobs",
    "Deloitte":     "https://apply.deloitte.com/careers/SearchJobs/strategy",
    "EY Parthenon": "https://careers.ey.com/ey/search/?q=strategy+operations",
    "Bain":         "https://www.bain.com/careers/find-a-role/",
    "Barclays":     "https://search.jobs.barclays/search-jobs/London/22160/4",
    "HSBC":         "https://mycareer.hsbc.com/en_GB/external/SearchJobs",
    "Accenture":    "https://www.accenture.com/gb-en/careers/jobsearch",
    "BlackRock":    "https://careers.blackrock.com/",
    "Fidelity":     "https://careers.fidelityinternational.com/",
}

NON_UK = [
    "united states", " us,", "canada", "australia", "japan",
    "singapore", "india", "germany", "france", "spain", "italy",
    "mexico", "brazil", "new york", "san francisco", "seattle",
    "tokyo", "sydney", "toronto", "paris", "berlin", "barcelona",
    "rome", "gurugram", "munich", "mÃ¼nchen", "dublin", "ireland",
    "portugal", "sweden", "netherlands", "belgium", "poland",
    "hyderabad", "delhi", "sofia", "riga", "warsaw", "amsterdam",
    "madrid", "milan", "shanghai", "beijing", "chicago",
]

def is_uk_loc(loc):
    if not loc: return True
    return not any(x in loc.lower() for x in NON_UK)

print("Checking company pages...")
print("-" * 55)

for company, token in GREENHOUSE.items():
    for base_url in [
        f"https://boards-api.greenhouse.io/v1/boards/{token}/jobs?content=true",
        f"https://boards.greenhouse.io/{token}",
    ]:
        try:
            r = request_with_backoff(base_url, headers=headers_req, timeout=12)
            r.raise_for_status()
            found = 0
            if "api" in base_url:
                for job in r.json().get("jobs", []):
                    title = job.get("title", "")
                    offices = job.get("offices", [])
                    location = offices[0].get("name", "") if offices else ""
                    if not is_uk_loc(location): continue
                    # High recall: do not require title keyword match before AI review
                    if title_is_excluded(title): continue
                    company_jobs.append({
                        "title": title, "company": company,
                        "location": location or "UK",
                        "salary": "See listing", "posted": "Live now",
                        "desc": clean_description(job.get("content", "")) or title,
                        "description_available": "Yes" if job.get("content") else "No",
                        "description_quality": "Full API" if job.get("content") else "Title Only",
                        "source": "Career Page",
                        "url": job.get("absolute_url", ""),
                    })
                    found += 1
            else:
                soup = BeautifulSoup(r.text, "html.parser")
                for a in soup.find_all("a", href=True):
                    title = a.get_text(strip=True)
                    href = a["href"]
                    if len(title) < 8 or len(title) > 110: continue
                    # High recall: do not require title keyword match before AI review
                    if title_is_excluded(title): continue
                    full_url = href if href.startswith("http") else f"https://boards.greenhouse.io{href}"
                    company_jobs.append({
                        "title": title, "company": company,
                        "location": "UK", "salary": "See listing",
                        "posted": "Live now", "desc": title,
                        "source": "Career Page", "url": full_url,
                    })
                    found += 1
            print(f"  âœ… {company:<25} {found} roles")
            break
        except:
            continue
    time.sleep(0.3)

for company, slug in LEVER.items():
    url = f"https://api.lever.co/v0/postings/{slug}?mode=json"
    try:
        r = request_with_backoff(url, headers=headers_req, timeout=12)
        r.raise_for_status()
        found = 0
        for job in r.json():
            title = job.get("text", "")
            location = job.get("categories", {}).get("location", "")
            if not is_uk_loc(location): continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            company_jobs.append({
                "title": title, "company": company,
                "location": location or "UK",
                "salary": "See listing", "posted": "Live now",
                "desc": clean_description(job.get("descriptionPlain") or job.get("description") or "") or title,
                "description_available": "Yes" if (job.get("descriptionPlain") or job.get("description")) else "No",
                "description_quality": "Full API" if (job.get("descriptionPlain") or job.get("description")) else "Title Only",
                "source": "Career Page",
                "url": job.get("hostedUrl", ""),
            })
            found += 1
        print(f"  âœ… {company:<25} {found} roles")
    except:
        print(f"  âŒ {company:<25} could not reach")
    time.sleep(0.3)

for company, url in DIRECT.items():
    try:
        r = request_with_backoff(url, headers=headers_req, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")
        found = 0
        for a in soup.find_all("a", href=True):
            title = a.get_text(strip=True)
            href = a["href"]
            if len(title) < 8 or len(title) > 110: continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            full_url = href if href.startswith("http") else url.split("/")[0] + "//" + url.split("/")[2] + href
            company_jobs.append({
                "title": title, "company": company,
                "location": "London/UK", "salary": "See listing",
                "posted": "Live now", "desc": title,
                "source": "Career Page", "url": full_url,
            })
            found += 1
        print(f"  âœ… {company:<25} {found} roles")
    except:
        print(f"  âŒ {company:<25} could not reach")
    time.sleep(0.5)

# (Removed: hardcoded placeholder roles from the previous candidate's build.)
print(f"âœ… Total company jobs: {len(company_jobs)}")

# â”€â”€ LinkedIn â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
HEADERS_LI = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
linkedin_jobs = []
seen_li = set()
LI_SEARCHES = [
    "strategy+and+operations+manager", "business+operations+manager",
    "corporate+strategy+manager", "commercial+strategy+manager",
    "business+transformation+manager", "transformation+manager",
    "change+manager", "chief+of+staff",
    "strategic+initiatives+manager", "operational+excellence+manager",
    "commercial+manager+strategy", "growth+manager",
    "partnerships+manager", "programme+manager+strategy",
    "customer+strategy+manager", "customer+experience+manager",
    "management+consultant+London", "strategy+consultant+London",
    "infrastructure+strategy+manager", "infrastructure+advisory",
    "asset+strategy+manager", "capital+planning+manager",
    "investment+planning+manager", "energy+strategy+manager",
    "transport+strategy+manager", "economic+regulation+manager",
    "net+zero+strategy+manager",
]
print()
print("Pulling LinkedIn...")
print("-" * 55)
for kw in LI_SEARCHES:
    url = (f"https://www.linkedin.com/jobs/search/?keywords={kw}"
           f"&location=London%2C+United+Kingdom&f_TPR=r604800&f_E=4%2C5")
    try:
        r = request_with_backoff(url, headers=HEADERS_LI, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.find_all("div", class_=lambda x: x and "base-card" in str(x))
        found = 0
        for card in cards:
            title_tag = card.find("h3", class_=lambda x: x and "base-search-card__title" in str(x))
            title = title_tag.get_text(strip=True) if title_tag else ""
            company_tag = card.find("h4", class_=lambda x: x and "base-search-card__subtitle" in str(x))
            company = company_tag.get_text(strip=True) if company_tag else ""
            loc_tag = card.find("span", class_=lambda x: x and "job-search-card__location" in str(x))
            location = loc_tag.get_text(strip=True) if loc_tag else "London"
            link_tag = card.find("a", class_=lambda x: x and "base-card__full-link" in str(x))
            link = link_tag["href"].split("?")[0] if link_tag else ""
            if not title or not company or not link or link in seen_li: continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            seen_li.add(link)
            linkedin_jobs.append({
                "title": title, "company": company, "location": location,
                "salary": "See listing", "posted": "This week",
                "desc": title, "source": "LinkedIn", "url": link,
            })
            found += 1
        print(f"  {kw.replace('+', ' ')[:40]:<40} {found} roles")
        time.sleep(2)
    except:
        print(f"  {kw[:40]} blocked")
print(f"âœ… LinkedIn: {len(linkedin_jobs)} roles")

# â”€â”€ Score â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print()
print("Scoring all jobs...")
results = []
seen_urls = set()

MISMATCH_RULES = [
    {"companies": ["jp morgan", "jpmorgan", "goldman sachs", "morgan stanley",
                   "deutsche bank", "ubs", "citi", "bank of america",
                   "bnp paribas", "lazard", "rothschild", "evercore"],
     "titles": ["vice president", " vp ", "managing director", "cib",
                "quantitative", "structuring", "trading"]},
    {"companies": ["skanska", "bovis", "mace", "wsp", "aecom",
                   "balfour", "costain", "kier"],
     "titles": ["commercial manager", "senior commercial manager"]},
]

def fails_mismatch(title, company):
    t = title.lower()
    c = company.lower()
    for rule in MISMATCH_RULES:
        if any(kw in c for kw in rule["companies"]) and any(kw in t for kw in rule["titles"]):
            return True
    return False

for job in board_jobs + company_jobs + linkedin_jobs:
    url = job.get("url", "")
    if not url or url in seen_urls: continue
    seen_urls.add(url)
    if fails_mismatch(job["title"], job.get("company", "")): continue
    total, interview_prob, reason, stretch, bucket = score_job(
        job["title"], job.get("desc", ""), job.get("location", "")
    )
    if job.get("source") == "Career Page" and total >= 25:
        total = min(total + 20, 100)
        reason = reason + " | +20 direct boost"
    if total < 40: continue
    results.append({
        "Bucket": "TBC", "Score /100": total, "Stretch (1-10)": stretch,
        "Title": job["title"], "Company": job["company"],
        "Location": job.get("location", ""),
        "Salary": job.get("salary", "See listing"),
        "Posted": job.get("posted", ""), "Source": job["source"],
        "Why It Matches": reason, "Status": "To Review",
        "Apply Link": url,
    })

seen_tc = set()
deduped = []
for r in results:
    key = (r["Title"].lower().strip(), r["Company"].lower().strip())
    if key not in seen_tc:
        seen_tc.add(key)
        deduped.append(r)

print(f"Interim pass: {len(results)} scored rows, {len(deduped)} after title/company dedup.")
print("The final export cell re-runs collection, scoring, dedup and the Excel build.")


In [ ]:
# Old duplicate Excel export cell disabled.
# This prevents Colab from downloading an older workbook without the Networking Tracker tab.
print("Skipped old duplicate Excel export. Use the final downloaded workbook from the final cell.")


In [ ]:
import requests
from bs4 import BeautifulSoup
import time

headers_req = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}

ubs_jobs = []

# UBS Workday API endpoints to try
UBS_URLS = [
    "https://jobs.ubs.com/TGNewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=JobListing&noback=1#",
    "https://jobs.ubs.com/TGNewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012",
    "https://ubs.wd3.myworkdayjobs.com/UBS_Career/jobs",
    "https://ubs.wd3.myworkdayjobs.com/en-US/UBS_Career",
]

# Workday API format
WORKDAY_API_URLS = [
    "https://ubs.wd3.myworkdayjobs.com/wday/cxs/ubs/UBS_Career/jobs",
    "https://ubs.wd3.myworkdayjobs.com/wday/cxs/ubs/UBS_Career/jobs?offset=0&limit=20&searchText=strategy+operations+London",
]

RELEVANT = RELEVANT_TITLES  # candidate taxonomy from the settings cell

EXCLUDE = HARD_EXCLUDE      # context-aware filtering via title_is_excluded()

NON_UK_CHECK = [
    "united states", "new york", "zurich", "switzerland",
    "singapore", "hong kong", "frankfurt", "paris",
    "sydney", "tokyo", "toronto", "mumbai", "india",
]

def is_uk_ubs(loc):
    if not loc: return True
    return not any(x in loc.lower() for x in NON_UK_CHECK)

print("Checking UBS career pages...")
print("-" * 55)

# Try Workday API first
for url in WORKDAY_API_URLS:
    try:
        r = requests.post(url,
            headers={
                **headers_req,
                "Accept": "application/json",
                "Content-Type": "application/json",
            },
            json={"appliedFacets": {}, "limit": 20, "offset": 0, "searchText": "strategy operations London"},
            timeout=12
        )
        print(f"  Workday API: {url} â†’ {r.status_code}")
        if r.status_code == 200:
            data = r.json()
            jobs = data.get("jobPostings", [])
            print(f"    Jobs found: {len(jobs)}")
            for job in jobs:
                title = job.get("title", "")
                location = job.get("locationsText", "")
                job_url = "https://ubs.wd3.myworkdayjobs.com" + job.get("externalPath", "")
                if not any(w in title.lower() for w in RELEVANT): continue
                if title_is_excluded(title): continue
                if not is_uk_ubs(location): continue
                ubs_jobs.append({
                    "title": title, "company": "UBS",
                    "location": location, "salary": "See listing",
                    "posted": "Live now", "desc": title,
                    "source": "Career Page", "url": job_url,
                })
                print(f"    âœ… {title} â€” {location}")
    except Exception as e:
        print(f"  Workday API error: {e}")
    time.sleep(1)

# Try scraping directly
for url in UBS_URLS:
    try:
        r = requests.get(url, headers=headers_req, timeout=12)
        print(f"  Direct: {url[:60]} â†’ {r.status_code} | Size: {len(r.text)}")
        if r.status_code == 200 and len(r.text) > 1000:
            soup = BeautifulSoup(r.text, "html.parser")
            found = 0
            for a in soup.find_all("a", href=True):
                title = a.get_text(strip=True)
                href = a["href"]
                if len(title) < 8 or len(title) > 110: continue
                if not any(w in title.lower() for w in RELEVANT): continue
                if title_is_excluded(title): continue
                full_url = href if href.startswith("http") else f"https://jobs.ubs.com{href}"
                ubs_jobs.append({
                    "title": title, "company": "UBS",
                    "location": "London/UK", "salary": "See listing",
                    "posted": "Live now", "desc": title,
                    "source": "Career Page", "url": full_url,
                })
                found += 1
            print(f"    Jobs found via scrape: {found}")
    except Exception as e:
        print(f"  Direct error: {e}")
    time.sleep(1)

# Try specific London strategy search
UBS_SEARCH_URLS = [
    "https://jobs.ubs.com/TGNewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=JobListing&noback=1&SearchCriteria=strategy+operations&SearchLocation=London",
    "https://ubs.wd3.myworkdayjobs.com/en-US/UBS_Career?q=strategy+operations&locations=London",
]

for url in UBS_SEARCH_URLS:
    try:
        r = requests.get(url, headers=headers_req, timeout=12)
        print(f"  Search URL: {url[:60]} â†’ {r.status_code} | Size: {len(r.text)}")
        if r.status_code == 200:
            soup = BeautifulSoup(r.text, "html.parser")
            text_preview = soup.get_text()[:500]
            print(f"    Preview: {text_preview[:200]}")
    except Exception as e:
        print(f"  Search error: {e}")

print()
print("=" * 55)
print(f"UBS roles found: {len(ubs_jobs)}")
for j in ubs_jobs:
    print(f"  - {j['title']} â€” {j['location']}")

In [ ]:
import requests
import json

headers_workday = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json",
    "Content-Type": "application/json",
    "X-Calypso-CSRF-Token": "undefined",
}

# Correct Workday API format
WORKDAY_COMPANIES = {
    "UBS":           "https://ubs.wd3.myworkdayjobs.com/wday/cxs/ubs/UBS_Career/jobs",
    "JP Morgan":     "https://jpmc.fa.us2.oraclecloud.com/hcmRestApi/resources/latest/recruitingCEJobRequisitions",
    "NatWest":       "https://natwest.wd3.myworkdayjobs.com/wday/cxs/natwest/NatWestGroupCareers/jobs",
    "Lloyds":        "https://lbg.wd3.myworkdayjobs.com/wday/cxs/lbg/LBG_External/jobs",
    "Santander":     "https://santander.wd3.myworkdayjobs.com/wday/cxs/santander/SantanderCareers/jobs",
    "KPMG":          "https://kpmg.wd3.myworkdayjobs.com/wday/cxs/kpmg/KPMG_UK_Careers/jobs",
    "PwC":           "https://pwc.wd3.myworkdayjobs.com/wday/cxs/pwc/Global_Campus_Experienced/jobs",
    "Mastercard":    "https://mastercard.wd1.myworkdayjobs.com/wday/cxs/mastercard/CorporateCareers/jobs",
    "Visa":          "https://visa.wd1.myworkdayjobs.com/wday/cxs/visa/Visa_Careers/jobs",
    "Alvarez Marsal":"https://alvarezandmarsal.wd1.myworkdayjobs.com/wday/cxs/alvarezandmarsal/alvarezandmarsal/jobs",
    "McKinsey":      "https://mckinsey.wd1.myworkdayjobs.com/wday/cxs/mckinsey/McKinsey_Careers/jobs",
    "Oliver Wyman":  "https://mmc.wd1.myworkdayjobs.com/wday/cxs/mmc/Oliver_Wyman_Careers/jobs",
    "Kearney":       "https://kearney.wd1.myworkdayjobs.com/wday/cxs/kearney/Kearney_Careers/jobs",
}

SEARCH_TERMS = [
    "strategy operations", "business transformation", "chief of staff",
    "commercial strategy", "infrastructure strategy", "asset strategy",
    "capital planning", "economic regulation",
]

def is_uk_role(loc):
    if not loc: return True
    return not any(x in loc.lower() for x in NON_UK)

workday_jobs = []
print("Testing Workday API for all companies...")
print("-" * 55)

for company, base_url in WORKDAY_COMPANIES.items():
    found_any = False
    for search_term in SEARCH_TERMS[:2]:  # Test first 2 terms
        payload = {
            "appliedFacets": {},
            "limit": 20,
            "offset": 0,
            "searchText": search_term,
        }
        try:
            r = requests.post(base_url, headers=headers_workday,
                            json=payload, timeout=12)
            if r.status_code == 200:
                data = r.json()
                jobs = data.get("jobPostings", [])
                found = 0
                for job in jobs:
                    title = job.get("title", "")
                    location = job.get("locationsText", "")
                    path = job.get("externalPath", "")
                    domain = "/".join(base_url.split("/")[:3])
                    job_url = domain + path
                    if not any(w in title.lower() for w in RELEVANT): continue
                    if title_is_excluded(title): continue
                    if not is_uk_role(location): continue
                    workday_jobs.append({
                        "title": title, "company": company,
                        "location": location, "salary": "See listing",
                        "posted": "Live now", "desc": title,
                        "source": "Career Page", "url": job_url,
                    })
                    found += 1
                if jobs:
                    print(f"  âœ… {company:<20} API works | Total jobs: {len(jobs)} | Relevant: {found}")
                    found_any = True
                    break
            elif r.status_code == 422:
                # Try GET instead of POST
                get_url = f"{base_url}?searchText={search_term.replace(' ', '+')}&limit=20"
                r2 = requests.get(get_url, headers=headers_workday, timeout=12)
                if r2.status_code == 200:
                    data = r2.json()
                    jobs = data.get("jobPostings", [])
                    print(f"  âœ… {company:<20} GET works | Jobs: {len(jobs)}")
                    found_any = True
                    break
                else:
                    print(f"  âŒ {company:<20} {r.status_code} POST, {r2.status_code} GET")
                    break
            else:
                print(f"  âŒ {company:<20} {r.status_code}")
                break
        except Exception as e:
            print(f"  âŒ {company:<20} Error: {str(e)[:50]}")
            break

print()
print("=" * 55)
print(f"Workday roles found: {len(workday_jobs)}")
for j in workday_jobs[:20]:
    print(f"  - {j['title']} @ {j['company']} â€” {j['location']}")

In [ ]:
import requests
import time

headers_workday = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

WORKDAY_WORKING = {
    "Santander":  "https://santander.wd3.myworkdayjobs.com/wday/cxs/santander/SantanderCareers/jobs",
    "Mastercard": "https://mastercard.wd1.myworkdayjobs.com/wday/cxs/mastercard/CorporateCareers/jobs",
}

# Try other Workday slugs with corrected formats
WORKDAY_RETRY = {
    "UBS":        "https://ubs.wd3.myworkdayjobs.com/wday/cxs/ubs/UBS_Career/jobs",
    "NatWest":    "https://natwest.wd3.myworkdayjobs.com/wday/cxs/natwest/NatWestGroupCareers/jobs",
    "KPMG":       "https://kpmg.wd3.myworkdayjobs.com/wday/cxs/kpmg/KPMG_UK_Careers/jobs",
    "Visa":       "https://visa.wd1.myworkdayjobs.com/wday/cxs/visa/Visa_Careers/jobs",
    "McKinsey":   "https://mckinsey.wd1.myworkdayjobs.com/wday/cxs/mckinsey/McKinsey_Careers/jobs",
    "Kearney":    "https://kearney.wd1.myworkdayjobs.com/wday/cxs/kearney/Kearney_Careers/jobs",
    "Lloyds":     "https://lbg.wd3.myworkdayjobs.com/wday/cxs/lbg/LBG_External/jobs",
    "PwC":        "https://pwc.wd3.myworkdayjobs.com/wday/cxs/pwc/Global_Campus_Experienced/jobs",
    "Oliver Wyman": "https://mmc.wd1.myworkdayjobs.com/wday/cxs/mmc/Oliver_Wyman_Careers/jobs",
    "Barclays":   "https://barclays.wd3.myworkdayjobs.com/wday/cxs/barclays/External/jobs",
    "HSBC":       "https://hsbc.wd3.myworkdayjobs.com/wday/cxs/hsbc/HSBCGlobalCareers/jobs",
    "Goldman":    "https://goldmansachs.wd1.myworkdayjobs.com/wday/cxs/goldmansachs/EmployeeJobPostings/jobs",
    "JP Morgan":  "https://jpmc.wd1.myworkdayjobs.com/wday/cxs/jpmc/JPMorganChase/jobs",
    "Deloitte":   "https://deloitte.wd1.myworkdayjobs.com/wday/cxs/deloitte/DTL_External/jobs",
    "BCG":        "https://bcg.wd3.myworkdayjobs.com/wday/cxs/bcg/BCG_Career_Internal_Staff_-_0/jobs",
    "Accenture":  "https://accenture.wd3.myworkdayjobs.com/wday/cxs/accenture/AccentureCareers/jobs",
}

RELEVANT = RELEVANT_TITLES  # candidate taxonomy from the settings cell
EXCLUDE = HARD_EXCLUDE      # context-aware filtering via title_is_excluded()

UK_LOCATIONS = ["london", "uk", "united kingdom", "england",
                "remote", "hybrid", "manchester", "edinburgh",
                "birmingham", "bristol", "leeds"]

def is_uk_role(loc):
    if not loc: return True
    loc_lower = loc.lower()
    # Must contain a UK location indicator
    return any(x in loc_lower for x in UK_LOCATIONS)

workday_jobs = []

# â”€â”€ Pull from confirmed working APIs with London filter â”€â”€â”€â”€â”€â”€â”€
print("Pulling from confirmed Workday APIs...")
print("-" * 55)

SEARCH_TERMS = [
    "strategy operations", "business transformation", "chief of staff",
    "commercial strategy", "infrastructure strategy", "asset strategy",
    "capital planning", "economic regulation",
]

for company, base_url in WORKDAY_WORKING.items():
    found_total = 0
    for search_term in SEARCH_TERMS:
        payload = {
            "appliedFacets": {},
            "limit": 20,
            "offset": 0,
            "searchText": search_term,
        }
        try:
            r = requests.post(base_url, headers=headers_workday,
                            json=payload, timeout=12)
            if r.status_code == 200:
                data = r.json()
                jobs = data.get("jobPostings", [])
                for job in jobs:
                    title = job.get("title", "")
                    location = job.get("locationsText", "")
                    path = job.get("externalPath", "")
                    domain = "/".join(base_url.split("/")[:3])
                    job_url = domain + path
                    if not any(w in title.lower() for w in RELEVANT): continue
                    if title_is_excluded(title): continue
                    if not is_uk_role(location): continue
                    # Avoid duplicates
                    if any(j["url"] == job_url for j in workday_jobs): continue
                    workday_jobs.append({
                        "title": title, "company": company,
                        "location": location, "salary": "See listing",
                        "posted": "Live now", "desc": title,
                        "source": "Career Page", "url": job_url,
                    })
                    found_total += 1
        except Exception as e:
            pass
        time.sleep(0.5)
    print(f"  âœ… {company:<20} {found_total} UK relevant roles")

# â”€â”€ Retry failed companies with corrected slugs â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print()
print("Retrying failed companies with corrected Workday slugs...")
print("-" * 55)

for company, base_url in WORKDAY_RETRY.items():
    try:
        payload = {
            "appliedFacets": {},
            "limit": 20,
            "offset": 0,
            "searchText": "strategy operations London",
        }
        r = requests.post(base_url, headers=headers_workday,
                         json=payload, timeout=12)
        if r.status_code == 200:
            data = r.json()
            jobs = data.get("jobPostings", [])
            found = 0
            for job in jobs:
                title = job.get("title", "")
                location = job.get("locationsText", "")
                path = job.get("externalPath", "")
                domain = "/".join(base_url.split("/")[:3])
                job_url = domain + path
                if not any(w in title.lower() for w in RELEVANT): continue
                if title_is_excluded(title): continue
                if not is_uk_role(location): continue
                if any(j["url"] == job_url for j in workday_jobs): continue
                workday_jobs.append({
                    "title": title, "company": company,
                    "location": location, "salary": "See listing",
                    "posted": "Live now", "desc": title,
                    "source": "Career Page", "url": job_url,
                })
                found += 1
            print(f"  âœ… {company:<20} {r.status_code} | {len(jobs)} total | {found} UK relevant")
        else:
            print(f"  âŒ {company:<20} {r.status_code}")
    except Exception as e:
        print(f"  âŒ {company:<20} Error: {str(e)[:40]}")
    time.sleep(0.5)

print()
print("=" * 55)
print(f"Total Workday UK roles: {len(workday_jobs)}")
print()
print("Roles found:")
for j in workday_jobs:
    print(f"  [{j['company']}] {j['title']} â€” {j['location']}")

In [ ]:
import requests
import time

# Fix 422 errors â€” Workday requires CSRF token
# Step 1: Get CSRF token from the page first, then use it in API call

headers_browser = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-GB,en;q=0.9",
}

CSRF_COMPANIES = {
    "UBS":       ("https://ubs.wd3.myworkdayjobs.com/en-US/UBS_Career",
                  "https://ubs.wd3.myworkdayjobs.com/wday/cxs/ubs/UBS_Career/jobs"),
    "NatWest":   ("https://natwest.wd3.myworkdayjobs.com/en-GB/NatWestGroupCareers",
                  "https://natwest.wd3.myworkdayjobs.com/wday/cxs/natwest/NatWestGroupCareers/jobs"),
    "KPMG":      ("https://kpmg.wd3.myworkdayjobs.com/en-GB/KPMG_UK_Careers",
                  "https://kpmg.wd3.myworkdayjobs.com/wday/cxs/kpmg/KPMG_UK_Careers/jobs"),
    "HSBC":      ("https://hsbc.wd3.myworkdayjobs.com/en-GB/HSBCGlobalCareers",
                  "https://hsbc.wd3.myworkdayjobs.com/wday/cxs/hsbc/HSBCGlobalCareers/jobs"),
    "Goldman":   ("https://goldmansachs.wd1.myworkdayjobs.com/en-US/EmployeeJobPostings",
                  "https://goldmansachs.wd1.myworkdayjobs.com/wday/cxs/goldmansachs/EmployeeJobPostings/jobs"),
    "JP Morgan": ("https://jpmc.wd1.myworkdayjobs.com/en-US/JPMorganChase",
                  "https://jpmc.wd1.myworkdayjobs.com/wday/cxs/jpmc/JPMorganChase/jobs"),
    "Deloitte":  ("https://deloitte.wd1.myworkdayjobs.com/en-US/DTL_External",
                  "https://deloitte.wd1.myworkdayjobs.com/wday/cxs/deloitte/DTL_External/jobs"),
    "BCG":       ("https://bcg.wd3.myworkdayjobs.com/en-US/BCG_Career_Internal_Staff_-_0",
                  "https://bcg.wd3.myworkdayjobs.com/wday/cxs/bcg/BCG_Career_Internal_Staff_-_0/jobs"),
    "McKinsey":  ("https://mckinsey.wd1.myworkdayjobs.com/en-US/McKinsey_Careers",
                  "https://mckinsey.wd1.myworkdayjobs.com/wday/cxs/mckinsey/McKinsey_Careers/jobs"),
    "Accenture": ("https://accenture.wd3.myworkdayjobs.com/en-US/AccentureCareers",
                  "https://accenture.wd3.myworkdayjobs.com/wday/cxs/accenture/AccentureCareers/jobs"),
    "Visa":      ("https://visa.wd1.myworkdayjobs.com/en-US/Visa_Careers",
                  "https://visa.wd1.myworkdayjobs.com/wday/cxs/visa/Visa_Careers/jobs"),
    "Kearney":   ("https://kearney.wd1.myworkdayjobs.com/en-US/Kearney_Careers",
                  "https://kearney.wd1.myworkdayjobs.com/wday/cxs/kearney/Kearney_Careers/jobs"),
}

RELEVANT = RELEVANT_TITLES  # candidate taxonomy from the settings cell
EXCLUDE = HARD_EXCLUDE      # context-aware filtering via title_is_excluded()
UK_LOCATIONS = [
    "london", "uk", "united kingdom", "england",
    "remote", "hybrid", "manchester", "edinburgh",
    "birmingham", "bristol", "leeds", "sheffield",
]

def is_uk_role(loc):
    if not loc: return True
    return any(x in loc.lower() for x in UK_LOCATIONS)

csrf_jobs = []
print("Getting CSRF tokens and pulling Workday APIs...")
print("-" * 55)

session = requests.Session()

for company, (page_url, api_url) in CSRF_COMPANIES.items():
    try:
        # Step 1: Visit the careers page to get cookies + CSRF token
        page_r = session.get(page_url, headers=headers_browser, timeout=12)

        # Extract CSRF token from cookies or response
        csrf_token = None
        for cookie in session.cookies:
            if "csrf" in cookie.name.lower() or "token" in cookie.name.lower():
                csrf_token = cookie.value
                break

        # Also check response headers
        if not csrf_token:
            csrf_token = page_r.headers.get("X-Calypso-CSRF-Token", "undefined")

        # Step 2: Call API with CSRF token
        api_headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Accept": "application/json",
            "Content-Type": "application/json",
            "X-Calypso-CSRF-Token": csrf_token or "undefined",
            "Referer": page_url,
            "Origin": "/".join(page_url.split("/")[:3]),
        }

        for search_term in ["strategy operations London", "business transformation London", "infrastructure strategy London"]:
            payload = {
                "appliedFacets": {},
                "limit": 20,
                "offset": 0,
                "searchText": search_term,
            }
            api_r = session.post(api_url, headers=api_headers,
                                json=payload, timeout=12)

            if api_r.status_code == 200:
                data = api_r.json()
                jobs = data.get("jobPostings", [])
                found = 0
                for job in jobs:
                    title = job.get("title", "")
                    location = job.get("locationsText", "")
                    path = job.get("externalPath", "")
                    domain = "/".join(api_url.split("/")[:3])
                    job_url = domain + path
                    if not any(w in title.lower() for w in RELEVANT): continue
                    if title_is_excluded(title): continue
                    if not is_uk_role(location): continue
                    if any(j["url"] == job_url for j in csrf_jobs): continue
                    csrf_jobs.append({
                        "title": title, "company": company,
                        "location": location, "salary": "See listing",
                        "posted": "Live now", "desc": title,
                        "source": "Career Page", "url": job_url,
                    })
                    found += 1
                if jobs:
                    print(f"  âœ… {company:<15} {api_r.status_code} | {len(jobs)} total | {found} UK relevant | CSRF: {csrf_token[:10] if csrf_token else 'none'}")
                    break
            else:
                print(f"  âŒ {company:<15} {api_r.status_code} | CSRF: {csrf_token[:10] if csrf_token else 'none'}")
                break
            time.sleep(0.5)

    except Exception as e:
        print(f"  âŒ {company:<15} Error: {str(e)[:50]}")
    time.sleep(1)

# Add all Workday jobs to company_jobs
all_workday = workday_jobs + csrf_jobs
company_jobs += all_workday

print()
print("=" * 55)
print(f"New roles from CSRF fix:     {len(csrf_jobs)}")
print(f"Total Workday roles:         {len(all_workday)}")
print(f"Total company jobs now:      {len(company_jobs)}")
print()
if csrf_jobs:
    print("New roles found:")
    for j in csrf_jobs:
        print(f"  [{j['company']}] {j['title']} â€” {j['location']}")

In [ ]:
# Add company-specific LinkedIn searches for blocked Workday companies
TARGETED_LI = [
    "National+Grid+strategy+London",
    "SSE+strategy+manager",
    "Octopus+Energy+strategy+London",
    "Thames+Water+strategy+manager",
    "Network+Rail+strategy+London",
    "Heathrow+strategy+manager",
    "Transport+for+London+strategy",
    "Arup+infrastructure+advisory+London",
    "Mott+MacDonald+advisory+London",
    "Turner+Townsend+infrastructure+advisory",
    "Baringa+energy+strategy+London",
    "Ofgem+price+control+London",
    "Ofwat+strategy+London",
    "Barclays+transformation+London",
    "HSBC+strategy+operations+London",
    "McKinsey+associate+London",
]

HEADERS_LI = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

print("Pulling company-specific LinkedIn searches...")
print("-" * 55)

extra_li = []
seen_li_extra = set()

for kw in TARGETED_LI:
    url = (f"https://www.linkedin.com/jobs/search/?keywords={kw}"
           f"&location=London%2C+United+Kingdom&f_TPR=r604800")
    try:
        r = requests.get(url, headers=HEADERS_LI, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.find_all("div", class_=lambda x: x and "base-card" in str(x))
        found = 0
        for card in cards:
            title_tag = card.find("h3", class_=lambda x: x and "base-search-card__title" in str(x))
            title = title_tag.get_text(strip=True) if title_tag else ""
            company_tag = card.find("h4", class_=lambda x: x and "base-search-card__subtitle" in str(x))
            company = company_tag.get_text(strip=True) if company_tag else ""
            loc_tag = card.find("span", class_=lambda x: x and "job-search-card__location" in str(x))
            location = loc_tag.get_text(strip=True) if loc_tag else "London"
            link_tag = card.find("a", class_=lambda x: x and "base-card__full-link" in str(x))
            link = link_tag["href"].split("?")[0] if link_tag else ""
            if not title or not company or not link: continue
            if link in seen_li_extra: continue
            # High recall: do not require title keyword match before AI review
            if title_is_excluded(title): continue
            seen_li_extra.add(link)
            extra_li.append({
                "title": title, "company": company,
                "location": location, "salary": "See listing",
                "posted": "This week", "desc": title,
                "source": "LinkedIn", "url": link,
            })
            found += 1
        print(f"  {kw.replace('+', ' ')[:45]:<45} {found} roles")
        time.sleep(2)
    except Exception as e:
        print(f"  {kw[:45]} blocked")

# Add to linkedin_jobs
linkedin_jobs += extra_li

print("-" * 55)
print(f"Extra LinkedIn roles: {len(extra_li)}")
print(f"Total LinkedIn roles: {len(linkedin_jobs)}")


In [ ]:
# ── ADDITIONAL SOURCES ───────────────────────────────────────────────────────
# Beyond Adzuna, Reed, LinkedIn and the Greenhouse/Lever/Workday readers above.
#
# Two kinds are added here:
#   1. Keyless applicant-tracking APIs (Workable, Ashby, SmartRecruiters,
#      Recruitee). These return clean JSON with a job-specific URL and the full
#      advert body, so they are the most reliable non-keyed sources available.
#   2. Public job pages read through their schema.org JobPosting markup, with a
#      link harvest as a fallback.
#
# Every source is independent: one failing never stops the others, and each
# reports what it actually returned in the summary at the end of this cell.
#
# EMPLOYER SLUGS: a board slug is the company's own id on that ATS, visible in
# its careers URL (e.g. apply.workable.com/<slug>/). The ones below are starting
# points - an unknown slug simply returns nothing and is reported as such. Add
# your own as you find them; that is the cheapest way to improve this notebook.
import time
import json as _json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# These normally come from the collection cells above. Fallbacks are defined so
# this cell does not depend on the order it happens to be run in.
if "headers_req" not in globals():
    headers_req = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                                 "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
if "company_jobs" not in globals():
    company_jobs = []
if "request_with_backoff" not in globals():
    def request_with_backoff(url, retries=3, base_sleep=1.0, **kwargs):
        for attempt in range(retries):
            try:
                response = requests.get(url, **kwargs)
                if response.status_code in (429, 500, 502, 503, 504):
                    raise requests.HTTPError(f"Retryable status {response.status_code}")
                return response
            except Exception:
                if attempt == retries - 1:
                    raise
                time.sleep(base_sleep * (2 ** attempt))

EXTRA_SOURCE_STATUS = {}


def _note_extra(name, count, detail=""):
    EXTRA_SOURCE_STATUS[name] = {"roles": count, "detail": detail}
    print(f"  {'ok  ' if count else 'none'}  {name:<34} {count:>4}  {detail}")


def _looks_relevant(title):
    """Cheap pre-filter. score_job() still makes the real decision later."""
    t = normalise_text(title)
    if title_is_excluded(title):
        return False
    return any(word in t for word in RELEVANT_TITLES)


def _record(title, company, location, desc, url, source, salary="", posted=""):
    return {
        "title": title, "company": company, "location": location or "UK",
        "salary": salary or "See listing", "posted": posted or "Live now",
        "desc": desc or title,
        "description_available": "Yes" if desc else "No",
        "description_quality": "Full API" if desc else "Title Only",
        "source": source, "url": url,
    }


# ── 1. WORKABLE ──────────────────────────────────────────────────────────────
WORKABLE_EMPLOYERS = {
    # name: workable account slug
    "Turner & Townsend": "turnerandtownsend",
    "Baringa": "baringa",
    "Frontier Economics": "frontier-economics",
    "Cornwall Insight": "cornwall-insight",
}


def fetch_workable(name, slug):
    jobs = []
    try:
        response = request_with_backoff(
            f"https://apply.workable.com/api/v3/accounts/{slug}/jobs",
            headers=headers_req, timeout=20, retries=2)
        if response.status_code != 200:
            return jobs
        for item in response.json().get("results", []):
            title = item.get("title", "")
            if not _looks_relevant(title):
                continue
            location = ", ".join(filter(None, [
                (item.get("location") or {}).get("city", ""),
                (item.get("location") or {}).get("country", ""),
            ]))
            if not is_uk_loc(location):
                continue
            shortcode = item.get("shortcode", "")
            jobs.append(_record(
                title, name, location,
                clean_description(item.get("description", "")),
                f"https://apply.workable.com/{slug}/j/{shortcode}/",
                "Workable careers"))
    except Exception:
        pass
    return jobs


# ── 2. ASHBY ─────────────────────────────────────────────────────────────────
ASHBY_EMPLOYERS = {
    "Octopus Energy Group": "octopusenergy",
    "Modo Energy": "modoenergy",
}


def fetch_ashby(name, slug):
    jobs = []
    try:
        response = request_with_backoff(
            f"https://api.ashbyhq.com/posting-api/job-board/{slug}?includeCompensation=true",
            headers=headers_req, timeout=20, retries=2)
        if response.status_code != 200:
            return jobs
        for item in response.json().get("jobs", []):
            title = item.get("title", "")
            if not _looks_relevant(title):
                continue
            location = item.get("location", "") or ""
            if not is_uk_loc(location):
                continue
            jobs.append(_record(
                title, name, location,
                clean_description(item.get("descriptionPlain") or item.get("descriptionHtml") or ""),
                item.get("jobUrl", ""), "Ashby careers"))
    except Exception:
        pass
    return jobs


# ── 3. SMARTRECRUITERS ───────────────────────────────────────────────────────
SMARTRECRUITERS_EMPLOYERS = {
    "National Grid": "NationalGrid",
    "Bosch": "Bosch",
    "Visa": "Visa",
}


def fetch_smartrecruiters(name, slug):
    jobs = []
    try:
        response = request_with_backoff(
            f"https://api.smartrecruiters.com/v1/companies/{slug}/postings?limit=100&country=uk",
            headers=headers_req, timeout=20, retries=2)
        if response.status_code != 200:
            return jobs
        for item in response.json().get("content", []):
            title = item.get("name", "")
            if not _looks_relevant(title):
                continue
            location_data = item.get("location", {}) or {}
            location = ", ".join(filter(None, [location_data.get("city", ""),
                                               location_data.get("country", "")]))
            if not is_uk_loc(location):
                continue
            posting_id = item.get("id", "")
            detail_text = ""
            try:
                detail = request_with_backoff(
                    f"https://api.smartrecruiters.com/v1/companies/{slug}/postings/{posting_id}",
                    headers=headers_req, timeout=20, retries=1)
                if detail.status_code == 200:
                    sections = ((detail.json().get("jobAd") or {}).get("sections") or {})
                    detail_text = clean_description(" ".join(
                        str((sections.get(part) or {}).get("text", ""))
                        for part in ("companyDescription", "jobDescription", "qualifications", "additionalInformation")))
            except Exception:
                pass
            jobs.append(_record(
                title, name, location, detail_text,
                item.get("ref", "") or f"https://jobs.smartrecruiters.com/{slug}/{posting_id}",
                "SmartRecruiters careers"))
            time.sleep(0.15)
    except Exception:
        pass
    return jobs


# ── 4. RECRUITEE ─────────────────────────────────────────────────────────────
RECRUITEE_EMPLOYERS = {
    # name: recruitee subdomain
    "Zopa Careers": "zopa",
}


def fetch_recruitee(name, slug):
    jobs = []
    try:
        response = request_with_backoff(
            f"https://{slug}.recruitee.com/api/offers/",
            headers=headers_req, timeout=20, retries=2)
        if response.status_code != 200:
            return jobs
        for item in response.json().get("offers", []):
            title = item.get("title", "")
            if not _looks_relevant(title):
                continue
            location = item.get("location", "") or item.get("city", "") or ""
            if not is_uk_loc(location):
                continue
            jobs.append(_record(
                title, name, location,
                clean_description(item.get("description", "")),
                item.get("careers_url", "") or item.get("url", ""), "Recruitee careers"))
    except Exception:
        pass
    return jobs


# ── 5. PUBLIC JOB PAGES VIA SCHEMA.ORG MARKUP ────────────────────────────────
# Most UK job sites embed JobPosting JSON-LD. Reading that is far more stable
# than CSS selectors, and it carries the full description. Sites that challenge
# automated requests simply return nothing and are reported.
PUBLIC_JOB_SITES = {
    "Totaljobs": "https://www.totaljobs.com/jobs/{kw}/in-london",
    "CV-Library": "https://www.cv-library.co.uk/{kw}-jobs-in-london",
    "Guardian Jobs": "https://jobs.theguardian.com/jobs/{kw}/",
    "Indeed UK": "https://uk.indeed.com/jobs?q={kw}&l=London&fromage=14",
    "Escape the City": "https://www.escapethecity.org/search?query={kw}",
}

PAGE_SEARCH_TERMS = [
    "strategy manager", "business operations manager", "transformation manager",
    "infrastructure strategy", "asset strategy manager", "capital planning manager",
]


def parse_jsonld_jobs(html, source, page_url=""):
    """Every schema.org JobPosting embedded in a page."""
    found = []
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception:
        return found
    for tag in soup.find_all("script", attrs={"type": "application/ld+json"}):
        raw = tag.string or tag.get_text() or ""
        if not raw.strip():
            continue
        try:
            data = _json.loads(raw)
        except Exception:
            continue
        stack = [data]
        while stack:
            node = stack.pop()
            if isinstance(node, list):
                stack.extend(node)
                continue
            if not isinstance(node, dict):
                continue
            if "@graph" in node:
                graph = node["@graph"]
                stack.extend(graph if isinstance(graph, list) else [graph])
            types = node.get("@type", "")
            types = types if isinstance(types, list) else [types]
            if not any(str(x).lower() == "jobposting" for x in types):
                continue
            org = node.get("hiringOrganization", {})
            company = org.get("name", "") if isinstance(org, dict) else str(org or "")
            location_node = node.get("jobLocation", {})
            if isinstance(location_node, list):
                location_node = location_node[0] if location_node else {}
            address = location_node.get("address", {}) if isinstance(location_node, dict) else {}
            location = ", ".join(filter(None, [
                str(address.get("addressLocality", "") or ""),
                str(address.get("addressRegion", "") or ""),
            ])) if isinstance(address, dict) else ""
            found.append(_record(
                str(node.get("title", "")), company or source, location,
                clean_description(node.get("description", "")),
                str(node.get("url", "") or page_url), source,
                posted=str(node.get("datePosted", ""))[:10]))
    return found


def harvest_job_links(html, base_url, source):
    """Fallback when a page has no structured data: title-only rows."""
    found = []
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception:
        return found
    seen_links = set()
    for anchor in soup.find_all("a", href=True):
        title = anchor.get_text(" ", strip=True)
        href = anchor["href"]
        if len(title) < 10 or len(title) > 110:
            continue
        if not any(hint in href.lower() for hint in ("/job/", "/jobs/", "/vacancy", "viewjob")):
            continue
        full_url = urljoin(base_url, href)
        if full_url in seen_links:
            continue
        seen_links.add(full_url)
        found.append(_record(title, "", "", "", full_url, source))
    return found


def fetch_public_site(name, template, terms):
    collected, seen_urls_site, structured = [], set(), 0
    for term in terms:
        url = template.format(kw=requests.utils.quote(term.replace(" ", "-")))
        try:
            response = request_with_backoff(url, headers=headers_req, timeout=20, retries=2)
            if response.status_code != 200 or len(response.text) < 500:
                continue
            rows = parse_jsonld_jobs(response.text, name, page_url=url)
            if rows:
                structured += len(rows)
            else:
                rows = harvest_job_links(response.text, url, name)
            for row in rows:
                key = row.get("url") or row.get("title")
                if not key or key in seen_urls_site:
                    continue
                if not _looks_relevant(row.get("title", "")):
                    continue
                seen_urls_site.add(key)
                collected.append(row)
        except Exception:
            continue
        finally:
            time.sleep(1.2)
    detail = f"{structured} with structured data" if structured else "no structured data / blocked"
    return collected, detail


# ── RUN THEM ─────────────────────────────────────────────────────────────────
extra_jobs = []
print("Additional sources")
print("-" * 72)

for ats_name, employers, fetcher in [
    ("Workable", WORKABLE_EMPLOYERS, fetch_workable),
    ("Ashby", ASHBY_EMPLOYERS, fetch_ashby),
    ("SmartRecruiters", SMARTRECRUITERS_EMPLOYERS, fetch_smartrecruiters),
    ("Recruitee", RECRUITEE_EMPLOYERS, fetch_recruitee),
]:
    total = 0
    reached = 0
    for employer_name, slug in employers.items():
        try:
            rows = fetcher(employer_name, slug)
        except Exception:
            rows = []
        if rows:
            reached += 1
        extra_jobs += rows
        total += len(rows)
        time.sleep(0.3)
    _note_extra(f"{ats_name} ({len(employers)} employers)", total,
                f"{reached} employers returned roles")

for site_name, template in PUBLIC_JOB_SITES.items():
    try:
        rows, detail = fetch_public_site(site_name, template, PAGE_SEARCH_TERMS)
    except Exception as error:
        rows, detail = [], f"failed: {str(error)[:40]}"
    extra_jobs += rows
    _note_extra(site_name, len(rows), detail)

# Feed them into the same pipeline as everything else.
company_jobs += extra_jobs
print("-" * 72)
print(f"Additional-source roles: {len(extra_jobs)}")
print(f"Company/extra pool now:  {len(company_jobs)}")


In [ ]:
import pandas as pd
from google.colab import files
from collections import Counter
import os
import json
import re
import requests

results = []
seen_urls = set()

STRICT_NON_UK_TERMS = [
    "united states", " usa", " us,", "u.s.", "canada", "australia", "japan",
    "singapore", "india", "germany", "france", "spain", "italy", "mexico", "brazil",
    "new york", "san francisco", "seattle", "boston", "los angeles", "washington",
    "chicago", "dallas", "miami", "austin", "tokyo", "sydney", "melbourne",
    "toronto", "paris", "berlin", "barcelona", "rome", "gurugram", "gurgaon",
    "mumbai", "bangalore", "bengaluru", "munich", "muenchen", "m?nchen",
    "frankfurt", "dublin", "ireland", "portugal", "lisbon", "sweden", "stockholm",
    "netherlands", "belgium", "poland", "hyderabad", "delhi", "sofia", "riga",
    "warsaw", "amsterdam", "madrid", "milan", "shanghai", "beijing", "hong kong",
    "seoul", "dubai", "abu dhabi", "riyadh", "apac", "emea", "latam",
    "remote - us", "remote us",
]

def strict_is_uk_loc(loc):
    loc_lower = normalise_text(loc)
    if not loc_lower:
        return True
    if any(term in loc_lower for term in STRICT_NON_UK_TERMS):
        return False
    country, _ = get_country_continent(loc_lower)
    if country in NON_UK_COUNTRIES:
        return False
    return True

is_uk_loc = strict_is_uk_loc


# ── UK Visa Sponsor Register ──────────────────────────────────────────────────
# Downloads the Home Office register of licensed Skilled Worker sponsors once
# per session and caches it in _uk_sponsor_cache. Falls back gracefully if the
# download fails so the rest of the pipeline is never blocked.
_uk_sponsor_cache = None

def load_uk_sponsor_register():
    """
    Downloads the Home Office register of licensed Skilled Worker sponsors.
    The asset URL changes with each publication, so we scrape the landing page
    for the current XLSX link rather than hardcoding it.
    """
    global _uk_sponsor_cache
    if _uk_sponsor_cache is not None:
        return _uk_sponsor_cache

    # The gov.uk page renders attachment links via JavaScript, so BeautifulSoup can't
    # find them. Use the GOV.UK Content API (plain JSON, no JS) instead.
    CONTENT_API = (
        "https://www.gov.uk/api/content/government/publications/"
        "register-of-licensed-sponsors-workers"
    )
    try:
        import io
        api_resp = requests.get(CONTENT_API, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
        api_resp.raise_for_status()
        content = api_resp.json()

        xlsx_link = None
        # details.documents is a list of HTML strings on gov.uk, not dicts
        import re as _re
        for doc in content.get("details", {}).get("documents", []):
            if isinstance(doc, dict):
                url_candidate = doc.get("url", "")
                if url_candidate.endswith(".xlsx"):
                    xlsx_link = url_candidate
                    break
            elif isinstance(doc, str):
                m = _re.search(r'https://assets\.publishing\.service\.gov\.uk[^"\'>\s]+\.xlsx', doc)
                if m:
                    xlsx_link = m.group(0)
                    break
        # Final fallback: scan the entire raw JSON response
        if not xlsx_link:
            matches = _re.findall(r'https://assets\.publishing\.service\.gov\.uk[^"\'>\s]+\.xlsx', api_resp.text)
            if matches:
                xlsx_link = matches[0]

        if not xlsx_link:
            print("  ⚠️  UK sponsor register: could not locate XLSX via GOV.UK Content API.")
            _uk_sponsor_cache = set()
            return _uk_sponsor_cache

        r = requests.get(xlsx_link, timeout=90, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        df = pd.read_excel(io.BytesIO(r.content), header=0)
        name_col = next(
            (c for c in df.columns if "organisation" in c.lower() or "name" in c.lower()),
            None,
        )
        if name_col:
            _uk_sponsor_cache = set(df[name_col].dropna().str.lower().str.strip())
            print(f"  ✅ UK sponsor register loaded: {len(_uk_sponsor_cache):,} companies")
        else:
            print(f"  ⚠️  UK sponsor register: unexpected columns {list(df.columns)[:5]}. Falling back.")
            _uk_sponsor_cache = set()
    except Exception as e:
        print(f"  ⚠️  UK sponsor register download failed: {e}. Text-only fallback active.")
        _uk_sponsor_cache = set()
    return _uk_sponsor_cache


def sponsorship_risk_enhanced(text, company=""):
    """
    Priority order:
      1. Explicit phrases in the job description.
      2. Cross-reference against the UK Home Office licensed sponsor register.
    """
    t = normalise_text(text)
    no_sponsorship_phrases = [
        "no sponsorship", "cannot sponsor", "unable to sponsor", "will not sponsor",
        "does not sponsor", "do not sponsor", "must have right to work",
        "right to work in the uk", "eligible to work in the uk", "existing right to work",
        "without sponsorship", "sponsorship is not available",
    ]
    positive_phrases = [
        "visa sponsorship", "sponsorship available", "skilled worker", "sponsor licence",
        "graduate visa",
    ]
    if any(phrase in t for phrase in no_sponsorship_phrases):
        return "High - right to work/no sponsorship wording"
    if any(phrase in t for phrase in positive_phrases):
        return "Low - sponsorship/visa friendly signal"

    register = load_uk_sponsor_register()
    if register and company:
        co_clean = normalise_text(company).replace(" (agency)", "").strip()
        if co_clean in register:
            return "Low - on UK sponsor register"
        co_tokens = [tok for tok in co_clean.split() if len(tok) > 3]
        if co_tokens and any(all(tok in entry for tok in co_tokens) for entry in register):
            return "Low - on UK sponsor register (partial match)"

    return "Unknown"


# ── CANDIDATE CAPABILITY MODEL ───────────────────────────────────────────────
# Dimensions map to what the CV evidences: data governance, data quality,
# regulatory reporting, reconciliation, MI/BI, business analysis, stakeholder
# management and project documentation.
CAPABILITY_DIMENSIONS = {
    "Commercial Strategy": ["commercial strategy", "commercial", "revenue", "growth",
                            "p&l", "margin", "pricing", "business case", "market"],
    "Business Operations": ["business operations", "operating model", "operational excellence",
                            "process", "efficiency", "service delivery", "capacity"],
    "Transformation & Change": ["transformation", "change management", "target operating model",
                                "redesign", "continuous improvement", "adoption"],
    "Root-Cause Diagnosis": ["root cause", "diagnostic", "analysis", "data-driven",
                             "segmentation", "insight", "investigation"],
    "Leadership & Scaling": ["lead a team", "team of", "line management", "direct reports",
                             "people management", "coaching", "capability", "attrition"],
    "Stakeholder & Exec Alignment": ["stakeholder", "executive", "board", "senior leadership",
                                     "influence", "cross-functional", "buy-in", "negotiation"],
    "Regulated Environment": ["regulated", "regulator", "regulatory", "compliance", "risk",
                              "audit", "governance", "controls", "assurance", "sla"],
    "Builder / From Scratch": ["from scratch", "greenfield", "establish", "set up", "playbook",
                               "framework", "sop", "design and implement", "no precedent"],
    "Banking & Financial Services": ["bank", "banking", "financial services", "payments",
                                     "lending", "credit", "fintech"],
    "Insurance & Bancassurance": ["insurance", "bancassurance", "protection", "life cover",
                                  "health cover", "motor", "renewals", "attachment"],
    "Infrastructure & Utilities": ["infrastructure", "energy", "utilities", "water", "rail",
                                   "transport", "networks", "grid", "net zero", "hydrogen"],
    "Economic Regulation": ["price control", "riio", "ofgem", "ofwat", "allowed revenue",
                            "regulatory submission", "economic regulation", "rab"],
    "AI Capability & Limits": ["ai", "artificial intelligence", "genai", "automation",
                               "machine learning", "decision support", "llm"],
    "Programme & Delivery": ["programme", "program", "pmo", "portfolio", "milestone",
                             "delivery", "raid", "roadmap", "agile"],
    "Customer & Proposition": ["customer", "customer experience", "customer journey",
                               "proposition", "service design", "cx"],
}



def score_capability_dimensions(job_row):
    text = normalise_text(" ".join([
        job_row.get("Title", ""), job_row.get("Company", ""),
        job_row.get("Job Description", ""), job_row.get("Why It Matches", ""),
    ]))
    scores = {}
    for dimension, keywords in CAPABILITY_DIMENSIONS.items():
        hits = sum(1 for keyword in keywords if keyword in text)
        # Two distinct signals in a dimension is already a strong indication.
        scores[dimension] = min(10, round((hits / 2) * 10, 1))
    return scores


# How central each capability is to this candidate's career, used to turn the
# dimension scores into a single "how close is this to the middle of her CV"
# rating. Financial Services Domain is applied separately as a modifier.
CAREER_ALIGNMENT_VALUE = {
    "Commercial Strategy": 10,
    "Business Operations": 10,
    "Transformation & Change": 10,
    "Root-Cause Diagnosis": 9,
    "Leadership & Scaling": 9,
    "Builder / From Scratch": 9,
    "Regulated Environment": 8,
    "Stakeholder & Exec Alignment": 8,
    "Banking & Financial Services": 8,
    "Infrastructure & Utilities": 8,
    "Economic Regulation": 8,
    "Insurance & Bancassurance": 7,
    "Programme & Delivery": 7,
    "Customer & Proposition": 7,
    "AI Capability & Limits": 6,
}



def capability_fit_score(dimensions):
    """
    Depth, not breadth. Averaging all fifteen dimensions punished a focused
    advert that matches the CV exactly, so only the strongest five count.
    """
    top = sorted(dimensions.values(), reverse=True)[:5]
    return round(sum(top) / max(1, len(top)), 1)


def career_alignment_score(dimensions):
    """
    Driven by the best-evidenced capabilities weighted by how central each one is
    to this CV, plus a financial-services modifier. A pure regulatory reporting
    role scores high even though it touches only one dimension.
    """
    ranked = sorted(
        (CAREER_ALIGNMENT_VALUE[dimension] * (score / 10.0)
         for dimension, score in dimensions.items() if dimension in CAREER_ALIGNMENT_VALUE),
        reverse=True,
    )
    best = ranked[0] if ranked else 0
    second = ranked[1] if len(ranked) > 1 else 0
    domain_modifier = 0.1 * dimensions.get("Financial Services Domain", 0)
    return min(10, round(0.75 * best + 0.25 * second + domain_modifier, 1))


def classify_job_family(job_row):
    text = normalise_text(" ".join([
        job_row.get("Title", ""), job_row.get("Job Description", ""), job_row.get("Why It Matches", ""),
    ]))
    families = {
        "Commercial Strategy & Operations": ["commercial strategy", "commercial operations",
                                             "business operations", "operating model",
                                             "commercial excellence", "revenue operations"],
        "Transformation & Change": ["transformation", "change management", "process redesign",
                                    "target operating model"],
        "BD & Partnerships": ["partnerships", "business development", "alliances",
                              "market expansion", "customer success"],
        "Builder / Enablement": ["enablement", "gtm operations", "playbook",
                                 "knowledge management", "from scratch", "capability building"],
        "Infrastructure & Utilities Strategy": ["infrastructure strategy", "asset strategy",
                                                "capital planning", "network strategy",
                                                "energy strategy", "utilities"],
        "Economic Regulation & Policy": ["price control", "riio", "ofgem", "ofwat",
                                         "economic regulation", "policy analyst",
                                         "regulatory strategy"],
        "AI-Adjacent Strategy & Ops": ["ai strategy", "ai operations", "ai adoption",
                                       "automation strategy", "decision intelligence"],
        "Insurance & Bancassurance": ["bancassurance", "insurance distribution",
                                      "insurance operations", "protection"],
        "Programme & PMO": ["programme", "program manager", "pmo", "portfolio", "delivery"],
        "Customer & Proposition": ["customer experience", "customer strategy", "proposition"],
        "Consulting": ["consultant", "consulting", "advisory", "engagement manager"],
        "Chief of Staff": ["chief of staff", "ceo office", "strategic initiatives"],
    }
    tags = [family for family, keywords in families.items() if any(keyword in text for keyword in keywords)]
    return ", ".join(tags[:4]) if tags else "General Analyst"


def classify_company_type(company):
    c = normalise_text(company)
    if any(x in c for x in ["national grid", "sse", "scottish power", "uk power networks",
                            "cadent", "northern powergrid", "octopus energy", "edf", "eon",
                            "e.on", "centrica", "drax", "ovo"]):
        return "Energy / Utilities"
    if any(x in c for x in ["thames water", "severn trent", "united utilities", "anglian water",
                            "yorkshire water", "southern water", "wessex water", "pennon"]):
        return "Water"
    if any(x in c for x in ["network rail", "hs2", "heathrow", "gatwick", "national highways",
                            "transport for london", "tfl", "avanti", "govia", "port of"]):
        return "Transport Infrastructure"
    if any(x in c for x in ["arup", "mott macdonald", "turner & townsend", "turner and townsend",
                            "aecom", "wsp", "jacobs", "atkins", "ramboll", "buro happold",
                            "frontier economics", "oxera", "baringa", "cornwall insight", "lcp"]):
        return "Infrastructure / Economic Advisory"
    if any(x in c for x in ["ofgem", "ofwat", "ofcom", "office of rail", "environment agency",
                            "nista", "infrastructure and projects authority", "department for",
                            "council", "government", "nhs", "civil service"]):
        return "Regulator / Public Sector"
    if any(x in c for x in ["barclays", "natwest", "hsbc", "lloyds", "santander", "ubs",
                            "jp morgan", "jpmorgan", "goldman", "citi", "nationwide",
                            "standard chartered", "bank of"]):
        return "Bank"
    if any(x in c for x in ["monzo", "wise", "revolut", "stripe", "adyen", "zopa", "tide",
                            "starling", "checkout", "klarna", "gocardless", "moneybox"]):
        return "Fintech"
    if any(x in c for x in ["mckinsey", "bcg", "bain", "oliver wyman", "kearney", "deloitte",
                            "pwc", "kpmg", "ey", "accenture", "slalom", "capgemini",
                            "alixpartners", "pa consulting", "north highland"]):
        return "Consulting"
    if any(x in c for x in ["aviva", "legal & general", "axa", "allianz", "zurich insurance",
                            "hiscox", "beazley", "direct line", "admiral", "insurance"]):
        return "Insurance"
    if any(x in c for x in [" fc", "football", "rugby", "cricket", "premier league", "sport"]):
        return "Sport & Entertainment"
    if any(x in c for x in ["google", "amazon", "microsoft", "meta", "salesforce", "oracle"]):
        return "Technology"
    if any(x in c for x in ["university", "college", "school"]):
        return "University"
    return "Scale-up" if any(x in c for x in ["limited", "ltd", "group"]) else "Other"


# Profile Bank section 10: what to lead with and what to downplay, by role type.
ROLE_PLAYBOOK = {
    "Commercial Strategy & Operations": (
        "Operating-model redesign, the segmentation and routing diagnoses, business cases "
        "carried through Risk, Finance and Ops",
        "Individual quota detail"),
    "Transformation & Change": (
        "Governance design, decision rights, influencing without authority, the DBAT "
        "business-side release involvement",
        "Individual sales-contribution stories"),
    "BD & Partnerships": (
        "Built from scratch, minimal oversight, the outsourced-partner renegotiation, "
        "auto-loan mining and renewals-as-growth",
        "AI projects"),
    "Builder / Enablement": (
        "Builds things himself: SOPs across all functions, KPI and forecasting frameworks "
        "where none existed, the Sports and Entertainment Club, the two AI platforms",
        "CERF, unless the role is research-adjacent"),
    "Infrastructure & Utilities Strategy": (
        "IERF and CERF named prominently, the NGET Conversion Gap measures, energy and "
        "utilities certifications, engineering identity",
        "Banking-first framing"),
    "Economic Regulation & Policy": (
        "The RIIO allowed-revenue insight, IERF's four-gate framework, regulated-environment "
        "stakeholder work with Risk and Compliance",
        "Sales framing"),
    "AI-Adjacent Strategy & Ops": (
        "The two self-built AI platforms and the dissertation on human-AI decision systems, "
        "framed as capability plus limits, not ML engineering",
        "CERF; never imply ML engineering credentials"),
    "Insurance & Bancassurance": (
        "250/160/120 percent figures, two years top performer, the auto-loan-mining idea, "
        "renewals as a growth channel, bancassurance vocabulary",
        "Non-insurance products; state the personal-versus-commercial-lines distinction plainly"),
    "Programme & PMO": (
        "Shift-coverage redesign through Risk, HR and leadership; onboarding drop-off 7 to 2 "
        "percent; AgilePM v3",
        "Individual sales numbers; never imply Scrum Master practical experience"),
    "Customer & Proposition": (
        "Customer drop-off root-cause reframe, journey redesign, proposition-to-segment "
        "realignment",
        "Team size, if the role reads as individual-contributor"),
    "Consulting": (
        "Moasure go-to-market project, CERF and IERF as published analytical work, "
        "structured problem solving",
        "Any implication of prior consulting-firm employment"),
    "Chief of Staff": (
        "Operating rhythms and KPI infrastructure built from nothing, executive business cases, "
        "cross-functional alignment",
        "Individual quota detail"),
}


def playbook_guidance(job_family, company_type=""):
    """Returns (lead_with, downplay) from the role-type playbook."""
    for family in [f.strip() for f in str(job_family or "").split(",")]:
        if family in ROLE_PLAYBOOK:
            return ROLE_PLAYBOOK[family]
    if company_type == "Sport & Entertainment":
        return ("Sports and Entertainment Club founding and the MBAT delegation, in a plain "
                "conversational register",
                "Consultancy jargon and career-arc polish")
    return ("One clear throughline: diagnose, build, secure buy-in, prove it with a number",
            "Narrow sector-specific certifications")


def recommend_resume(job_family, dimensions, track_text=""):
    """Which CV variant to lead with, and how much rework it honestly needs."""
    family_text = normalise_text(job_family)
    if "infrastructure" in family_text or "regulation" in family_text:
        resume = "Infrastructure / Regulation Resume (CERF + IERF forward)"
    elif "insurance" in family_text:
        resume = "Insurance Resume (bancassurance, personal lines)"
    elif "ai-adjacent" in family_text:
        resume = "AI-Adjacent Resume (platforms + dissertation forward)"
    elif "transformation" in family_text:
        resume = "Transformation Resume"
    elif "bd & partnerships" in family_text:
        resume = "Partnerships Resume (existing-base expansion)"
    elif "builder" in family_text:
        resume = "Builder / Enablement Resume"
    elif "consulting" in family_text:
        resume = "Consulting Resume (Moasure + published work)"
    elif "programme" in family_text or "pmo" in family_text:
        resume = "Programme Delivery Resume"
    else:
        resume = "Commercial Strategy Resume"

    changes = []
    if dimensions.get("Infrastructure & Utilities", 0) >= 5 or dimensions.get("Economic Regulation", 0) >= 5:
        changes.append("attach the 14-page CERF version; open on the sector evidence, not banking")
    if dimensions.get("Insurance & Bancassurance", 0) >= 5:
        changes.append("state personal lines versus commercial lines plainly")
    if dimensions.get("AI Capability & Limits", 0) >= 5:
        changes.append("lead with the built platforms; never imply ML engineering")
    if dimensions.get("Leadership & Scaling", 0) >= 5:
        changes.append("state the 28 to 71 scaling and the four Senior Team Leads early")
    if dimensions.get("Builder / From Scratch", 0) >= 5:
        changes.append("foreground SOPs, KPI frameworks and the club built from nothing")
    if dimensions.get("Regulated Environment", 0) >= 5:
        changes.append("surface Risk, Compliance and audit-facing supplier governance")
    if not changes:
        changes.append("minor headline and skills alignment only")

    track = normalise_text(track_text)
    if "name these gaps" in track:
        effort = "High"
    elif any(flag in track for flag in ("generic title", "no mapped track")):
        effort = "High"
    elif track.startswith("primary"):
        effort = "Low"
    elif track.startswith("secondary"):
        effort = "Medium"
    else:
        effort = "High"
    return resume, effort, "; ".join(changes[:4])


def networking_recommendation(job_row, company_type):
    """
    The Profile Bank is explicit that this is an access problem. Where a real
    contact or a citable paper exists, say so; otherwise name the warmest route.
    """
    company = normalise_text(job_row.get("Company", ""))
    angle = job_row.get("Evidence Angle", "")
    if "national grid" in company:
        return ("Amir (dissertation interviewee) can refer internally and has not refused when "
                "asked. Lead with the NGET Conversion Gap work.")
    if "cadent" in company:
        return ("Shivam Sharma at Cadent offered to vouch after application. Previous final-round "
                "feedback asked for clearer motivation to join Cadent specifically - address it.")
    if "barclays" in company:
        return "Matt Hammerstein is an existing conversation, not cold outreach."
    if "monzo" in company:
        return "Rupert Keeley call already arranged; an independent Monzo strategy assignment exists."
    if "warwick" in company or "wbs" in company:
        return "Sarah Jackson and Konstantina Dee are direct contacts; the Alumni Ambassador role is held."
    if angle:
        return f"Open with the published work rather than the CV. {angle}"
    if company_type in ("Energy / Utilities", "Water", "Transport Infrastructure",
                        "Infrastructure / Economic Advisory", "Regulator / Public Sector"):
        return ("Lead with CERF/IERF as the credibility bridge, then find a strategy, regulation "
                "or customer-strategy contact before applying.")
    if company_type == "Consulting":
        return ("Boutique and referral routes only; large firms with a hard prior-consulting "
                "requirement are treated as closed unless a referral exists.")
    if company_type == "Bank":
        return "Find a strategy, transformation or commercial operations contact inside the bank."
    if company_type == "Insurance":
        return "Approach a distribution or bancassurance contact; the personal-lines numbers are the hook."
    if company_type == "Sport & Entertainment":
        return "Use the Sports and Entertainment Club and Warwick Sport involvement as the genuine opener."
    return ("Volume applications die at the filter. Find one hiring-manager-adjacent contact, or "
            "lead with published work.")


def append_rule_decision_layer(shortlisted_jobs):
    enriched = []
    for row in shortlisted_jobs:
        dimensions = score_capability_dimensions(row)
        job_family = classify_job_family(row)
        company_type = classify_company_type(row.get("Company", ""))
        resume, resume_effort, resume_delta = recommend_resume(job_family, dimensions, row.get("Why It Matches", ""))
        lead_with, downplay = playbook_guidance(job_family, company_type)
        networking = networking_recommendation(row, company_type)

        rule_score = float(row.get("Score /100", 0) or 0)
        stretch = float(row.get("Stretch (1-10)", 10) or 10)
        capability_fit = capability_fit_score(dimensions)
        career_alignment = career_alignment_score(dimensions)

        # Profile Bank rule 21: the Graduate visa gives two years of open work rights
        # after the MBA, so no-sponsorship wording is not a blocker at the point of
        # hiring. No penalty is applied.
        sponsorship_penalty = 0
        # An existing contact or a citable paper is what actually produces interviews.
        evidence_bonus = 6 if row.get("Evidence Angle") else 0
        source_quality_bonus = {"Adzuna": 4, "Reed": 4, "Career Page": 3, "LinkedIn": 0}.get(row.get("Source", ""), 1)
        description_bonus = 4 if row.get("Description Available") == "Yes" else 0
        resume_effort_penalty = {"Low": 0, "Medium": 5, "High": 12}.get(resume_effort, 5)
        stretch_penalty = max(0, stretch - 5) * 3

        # Rows the source only gave a title for have no text for the capability
        # model to read. Lean on the rule score rather than punishing the row for
        # information the job board withheld.
        has_description = (
            row.get("Description Available") == "Yes"
            and not str(row.get("Description Quality", "")).startswith("Title Only")
        )
        rule_weight, capability_weight = (0.45, 0.20) if has_description else (0.85, 0.0)

        priority_score = round(
            rule_score * rule_weight +
            capability_fit * 10 * capability_weight +
            career_alignment * 10 * capability_weight +
            source_quality_bonus +
            description_bonus +
            evidence_bonus -
            resume_effort_penalty -
            stretch_penalty -
            sponsorship_penalty,
            1,
        )
        priority_score = max(0, min(100, priority_score))

        if priority_score >= 72:
            apply_decision = "YES"
            final_bucket = "A - Apply Now"
        elif priority_score >= 55:
            apply_decision = "MAYBE"
            final_bucket = "B - High Upside"
        elif priority_score >= 40:
            apply_decision = "NETWORK"
            final_bucket = "C - Network First"
        else:
            apply_decision = "NO"
            final_bucket = "D - Low Priority"

        row.update({
            "Job Family": job_family,
            "Company Type": company_type,
            "Capability Fit": capability_fit,
            "Career Alignment": career_alignment,
            "Interview Probability Final": round(priority_score, 1),
            "Resume to Use": resume,
            "Resume Changes Needed": resume_delta,
            "Resume Effort Required": resume_effort,
            "Cover Letter Effort Required": "Low" if apply_decision == "YES" and resume_effort == "Low" else "Medium",
            "Networking Recommendation": networking,
            "Lead With": lead_with,
            "Downplay": downplay,
            "Priority Score": priority_score,
            "Overall Priority": 0,
            "Apply?": apply_decision,
            "Final Bucket": final_bucket,
            "Decision Reasoning": f"Rule score {rule_score}/100; capability fit {capability_fit}/10; career alignment {career_alignment}/10; stretch {stretch}/10; description {row.get('Description Quality', 'Unknown')}" + ("" if has_description else " (title only - ranked on the rule score)") + f". {row.get('Why It Matches', '')}",
            "Capability Scores": "; ".join([f"{k}: {v}" for k, v in dimensions.items() if v >= 4]) or "No strong capability signals found in available text",
        })
        enriched.append(row)

    enriched.sort(key=lambda x: -x.get("Priority Score", 0))
    for index, row in enumerate(enriched, 1):
        row["Overall Priority"] = index
    return enriched


# ── FULL JOB DESCRIPTION HANDLING ────────────────────────────────────────────
# The Excel "Job Description" column holds the COMPLETE description returned by
# the source, after HTML stripping only. It is never word-trimmed, sentence-
# trimmed or replaced by an AI summary. The only length handling here exists
# because Excel itself refuses more than 32,767 characters in one cell, so the
# overflow spills into a continuation column instead of being thrown away.
EXCEL_CELL_LIMIT = 32000
_ILLEGAL_XLSX_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


def clean_for_excel(value):
    """Removes control characters openpyxl cannot write. No content is trimmed."""
    if value is None:
        return ""
    return _ILLEGAL_XLSX_CHARS.sub(" ", str(value))


def split_description_for_excel(text):
    """Returns (part_1, part_2). Both parts together are the full description."""
    text = clean_for_excel(text)
    if len(text) <= EXCEL_CELL_LIMIT:
        return text, ""
    remainder = text[EXCEL_CELL_LIMIT:]
    if len(remainder) > EXCEL_CELL_LIMIT:
        remainder = remainder[:EXCEL_CELL_LIMIT - 60] + " [...continues - open the Apply Link for the rest]"
    return text[:EXCEL_CELL_LIMIT], remainder


def describe_description_quality(desc, title, source, supplied_quality=""):
    """Honest labelling of what the source actually gave us."""
    text = (desc or "").strip()
    if supplied_quality:
        base = supplied_quality
    elif not text or text.lower() == (title or "").strip().lower():
        base = "Title Only"
    elif source in ("Adzuna", "Reed"):
        base = "Board Summary"
    elif source == "LinkedIn":
        base = "Title Only"
    else:
        base = "Full API"
    if text.endswith(("…", "...")) or text.endswith("… "):
        base += " (truncated by source)"
    return base


# ── EXCEL FORMATTING: CLICKABLE LINKS + USABILITY ────────────────────────────
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter

HYPERLINK_COLUMNS = {"Apply Link", "LinkedIn URL", "URL"}
WRAP_COLUMNS = {
    "Evidence Angle", "Honest Gaps", "Lead With", "Downplay",
    "Job Description", "Job Description (continued)", "Why It Matches",
    "Decision Reasoning", "Capability Scores", "Networking Recommendation",
    "Resume Changes Needed", "Notes",
}


def format_workbook(path):
    """
    Freezes headers, enables filters, wraps long text and turns every URL column
    into a real clickable Excel hyperlink. Cells without a usable URL are left
    blank - no placeholder links are invented.
    """
    workbook = load_workbook(path)
    link_font = Font(color="0563C1", underline="single")
    header_font = Font(bold=True)
    links_made = 0

    for worksheet in workbook.worksheets:
        if worksheet.max_row < 1:
            continue
        headers = [cell.value for cell in worksheet[1]]
        for cell in worksheet[1]:
            cell.font = header_font
            cell.alignment = Alignment(vertical="center", wrap_text=False)

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = f"A1:{get_column_letter(worksheet.max_column)}{worksheet.max_row}"

        has_wrapped_column = False
        for index, name in enumerate(headers, start=1):
            letter = get_column_letter(index)
            if name in HYPERLINK_COLUMNS:
                worksheet.column_dimensions[letter].width = 48
                for row in range(2, worksheet.max_row + 1):
                    cell = worksheet.cell(row=row, column=index)
                    url = str(cell.value or "").strip()
                    if url.lower().startswith(("http://", "https://")):
                        cell.hyperlink = url
                        cell.font = link_font
                        cell.alignment = Alignment(vertical="top")
                        links_made += 1
                    else:
                        cell.value = None
            elif name in WRAP_COLUMNS:
                has_wrapped_column = True
                worksheet.column_dimensions[letter].width = 80
                for row in range(2, worksheet.max_row + 1):
                    worksheet.cell(row=row, column=index).alignment = Alignment(wrap_text=True, vertical="top")
            else:
                worksheet.column_dimensions[letter].width = max(12, min(30, len(str(name or "")) + 6))

        # Keep rows a readable fixed height so wrapped descriptions do not
        # stretch a row over a whole screen. Expand a row to read it in full.
        if has_wrapped_column:
            for row in range(2, worksheet.max_row + 1):
                worksheet.row_dimensions[row].height = 42

    workbook.save(path)
    return links_made


# ── VISA POSITION OVERRIDE ───────────────────────────────────────────────────
# The generic reader above treats "no sponsorship" as a risk. For this candidate
# it is not: the Graduate visa gives two years of open work rights after the MBA.
# This reports the position instead of penalising it.
def sponsorship_risk_enhanced(text, company=""):
    t = normalise_text(text)
    no_sponsorship = [
        "no sponsorship", "cannot sponsor", "unable to sponsor", "will not sponsor",
        "does not sponsor", "do not sponsor", "without sponsorship",
        "sponsorship is not available",
    ]
    right_to_work = [
        "must have the right to work", "right to work in the uk",
        "existing right to work", "eligible to work in the uk",
    ]
    positive = ["visa sponsorship", "sponsorship available", "skilled worker",
                "sponsor licence", "graduate visa"]
    # Negative wording is checked first: "no sponsorship available" contains the
    # substring "sponsorship available", so the positive branch would swallow it.
    if any(phrase in t for phrase in no_sponsorship):
        return "No sponsorship stated - still viable on the Graduate visa; revisit at the two-year mark"
    if any(phrase in t for phrase in positive):
        return "Sponsors - viable now and beyond the Graduate visa"
    if any(phrase in t for phrase in right_to_work):
        return "Right to work required - satisfied by the Graduate visa"
    return "Not stated"


AGENCIES = [
    "reed", "hays", "robert half", "michael page", "recruitment", "talent", "staffing",
    "manpower", "adecco", "randstad", "3search", "tiger recruitment", "cedar",
    "run-time", "sphere", "carousel", "focus search", "brook street", "hexagon",
    "wild berry", "edge careers", "empro", "harnham", "eames", "morgan mckinley",
    "nigel frank", "oliver james",
]

# Company + title pairs that are a known mismatch for this CV: senior markets and
# front-office roles at the large banks, and engineering-heavy data roles at the
# data vendors.
MISMATCH_RULES = [
    {"companies": ["jp morgan", "jpmorgan", "goldman sachs", "morgan stanley", "deutsche bank",
                   "ubs", "citi", "bank of america", "bnp paribas", "lazard", "rothschild", "evercore"],
     "titles": ["vice president", " vp ", "managing director", "cib", "quantitative",
                "structuring", "trading", "sales trade", "investment banking"]},
    # Infrastructure contractors: "commercial manager" there means quantity surveying
    # and contract administration, which is not infrastructure strategy.
    {"companies": ["skanska", "bovis", "mace", "balfour", "costain", "kier", "laing o'rourke",
                   "morgan sindall", "galliford", "vinci", "bam ", "murphy"],
     "titles": ["commercial manager", "senior commercial manager", "quantity surveyor",
                "site manager", "project manager", "package manager"]},
    # Utilities and transport operators: field and network operations, not strategy.
    {"companies": ["national grid", "sse", "thames water", "severn trent", "united utilities",
                   "network rail", "national highways", "uk power networks", "cadent"],
     "titles": ["field", "technician", "operative", "linesman", "control room",
                "maintenance", "shift", "depot", "signaller"]},
]



def fails_mismatch(title, company):
    t = normalise_text(title)
    c = normalise_text(company)
    for rule in MISMATCH_RULES:
        if any(kw in c for kw in rule["companies"]) and any(kw in t for kw in rule["titles"]):
            return True
    return False


def make_bucket(score, stretch):
    if score >= 70 and stretch <= 5:
        return "A - Apply Now"
    if score >= 60 and stretch <= 7:
        return "B - High Upside"
    if score >= 50 and stretch <= 9:
        return "C - Network First"
    return "D - Skip"


# Pre-load the UK sponsor register once before scoring begins.
print("Loading UK visa sponsor register...")
load_uk_sponsor_register()

all_jobs = board_jobs + company_jobs + linkedin_jobs
removed_non_uk = 0
removed_mismatch = 0
removed_excluded = 0

for job in all_jobs:
    url = job.get("url", "")
    if not url or url in seen_urls:
        continue
    seen_urls.add(url)

    title = job.get("title", "")
    company = job.get("company", "")
    location = job.get("location", "")
    desc = job.get("desc", "")          # full text from the source, HTML already stripped
    source = job.get("source", "")

    description_available = job.get("description_available") or (
        "Yes" if desc and desc.strip() and desc.strip().lower() != title.strip().lower() else "No"
    )
    description_quality = describe_description_quality(
        desc, title, source, job.get("description_quality", "")
    )

    if location and not is_uk_loc(location):
        removed_non_uk += 1
        continue
    if fails_mismatch(title, company):
        removed_mismatch += 1
        continue

    total, track_points, reason, stretch, bucket = score_job(title, desc, location)

    if total == 0 and str(reason).upper().startswith("EXCLUDED"):
        removed_excluded += 1
        continue

    # Career pages and LinkedIn often return title-only rows, which starves the
    # content-based dimensions. The boost compensates for the missing text, so it
    # is largest exactly where the description is thinnest - and it is never
    # applied to a role that failed to map onto one of the candidate's tracks.
    if track_points > 0 and total >= 25:
        if source == "Career Page":
            boost = 20 if description_quality.startswith("Title Only") else 8
            total = min(total + boost, 100)
            reason += f" | +{boost} career-page boost"
        elif source == "LinkedIn":
            boost = 12 if description_quality.startswith("Title Only") else 5
            total = min(total + boost, 100)
            reason += f" | +{boost} LinkedIn limited-description boost"

    stretch = calculate_stretch(title, desc, company, total, track_points)
    bucket = make_bucket(total, stretch)

    country, continent = get_country_continent(location)
    if country in NON_UK_COUNTRIES:
        removed_non_uk += 1
        continue

    company_display = company
    if any(a in normalise_text(company_display) for a in AGENCIES) and "(Agency)" not in company_display:
        company_display += " (Agency)"

    description_main, description_overflow = split_description_for_excel(desc)

    risk_text = f"{title} {desc}"
    angle = evidence_angle(company, title, desc)
    named_gaps = honest_gaps(title, desc)
    results.append({
        "Evidence Angle": angle,
        "Honest Gaps": "; ".join(named_gaps) if named_gaps else "None identified",
        "Bucket": bucket,
        "Score /100": total,
        "Stretch (1-10)": stretch,
        "Title": title,
        "Company": company_display,
        "Location": location,
        "Country": country,
        "Continent": continent,
        "Salary": job.get("salary", "See listing"),
        "Posted": job.get("posted", ""),
        "Source": source,
        "Duplicate Sources": "",
        "Why It Matches": reason,
        "Job Description": description_main,
        "Job Description (continued)": description_overflow,
        "Description Available": description_available,
        "Description Quality": description_quality,
        "Sponsorship risk": sponsorship_risk_enhanced(risk_text, company_display),
        "Status": "To Review",
        "AI Remarks": "",
        "AI Review Decision": "",
        "Apply Link": url,
    })


# ── DEDUPLICATION: keep the best copy of a job, not the first one seen ───────
QUALITY_RANK = {"Full API": 3, "Board Summary": 2, "Title Only": 1, "Unavailable": 0}
SOURCE_RANK = {"Career Page": 3, "Adzuna": 2, "Reed": 2, "LinkedIn": 1}


def url_specificity(url):
    """A job-specific URL beats a generic careers landing page."""
    u = normalise_text(url)
    if not u:
        return 0
    if any(token in u for token in ["/job/", "/jobs/", "job_id", "jobid", "requisition", "/vacancy",
                                    "currentjobid", "/postings/", "viewjob", "-job-"]):
        return 3
    if u.rstrip("/").count("/") > 3:
        return 2
    return 1


def record_quality(row):
    """Ordering: completeness of the description, then URL specificity, then source."""
    quality_label = str(row.get("Description Quality", "")).split(" (")[0]
    return (
        QUALITY_RANK.get(quality_label, 1),
        len(row.get("Job Description", "") or "") + len(row.get("Job Description (continued)", "") or ""),
        url_specificity(row.get("Apply Link", "")),
        SOURCE_RANK.get(row.get("Source", ""), 1),
    )


best_by_key = {}
duplicate_sources = {}
for r in results:
    key = (normalise_text(r["Title"]), normalise_text(r["Company"]).replace(" (agency)", ""))
    duplicate_sources.setdefault(key, set()).add(r.get("Source", ""))
    incumbent = best_by_key.get(key)
    if incumbent is None:
        best_by_key[key] = r
        continue
    winner, loser = (r, incumbent) if record_quality(r) > record_quality(incumbent) else (incumbent, r)
    # Do not lose information the weaker copy had: fill any gaps from it.
    for field in ("Salary", "Posted", "Location", "Job Description", "Job Description (continued)",
                  "Apply Link", "Why It Matches"):
        if not str(winner.get(field, "") or "").strip() or str(winner.get(field, "")).strip() in ("See listing", ""):
            if str(loser.get(field, "") or "").strip():
                winner[field] = loser[field]
    winner["Score /100"] = max(winner.get("Score /100", 0), loser.get("Score /100", 0))
    best_by_key[key] = winner

deduped = []
for key, row in best_by_key.items():
    others = sorted(s for s in duplicate_sources.get(key, set()) if s and s != row.get("Source"))
    row["Duplicate Sources"] = ", ".join(others)
    deduped.append(row)

deduped.sort(key=lambda x: (x["Bucket"], -x["Score /100"], x["Stretch (1-10)"]))

# Production decision layer: deterministic rules rank every broadly collected
# candidate. The AI review cell is optional and is never called during this run.
deduped = append_rule_decision_layer(deduped)

a = [r for r in deduped if r["Bucket"] == "A - Apply Now"]
b = [r for r in deduped if r["Bucket"] == "B - High Upside"]
c = [r for r in deduped if r["Bucket"] == "C - Network First"]
print("=" * 55)
print(f"  Adzuna + Reed:       {len(board_jobs)}")
print(f"  Company pages:       {len(company_jobs)}")
print(f"  LinkedIn:            {len(linkedin_jobs)}")
print(f"  Removed non-UK:      {removed_non_uk}")
print(f"  Removed mismatches:  {removed_mismatch}")
print(f"  Removed excluded:    {removed_excluded}")
print(f"  Total unique:        {len(deduped)}")
print(f"  A - Apply Now:       {len(a)}")
print(f"  B - High Upside:     {len(b)}")
print(f"  C - Network First:   {len(c)}")
print("=" * 55)
print()
print("Bucket A - Apply Now:")
for i, r in enumerate(a, 1):
    print(f"  {i}. [{r['Score /100']}] Stretch:{r['Stretch (1-10)']} {r['Title']} @ {r['Company']} - {r['Location']}")

continent_counts = Counter(r["Continent"] for r in deduped)
country_counts = Counter(r["Country"] for r in deduped)
family_counts = Counter(r.get("Job Family", "Unknown") for r in deduped)
print("=" * 55)
print("Roles by Continent:")
for continent, count in sorted(continent_counts.items(), key=lambda x: -x[1]):
    print(f"  {continent:<20} {count}")
print()
print("Top Countries:")
for country, count in sorted(country_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {country:<20} {count}")
print()
print("Top Job Families:")
for family, count in sorted(family_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {family[:38]:<40} {count}")
print("=" * 55)

cols = [
    "Bucket", "Final Bucket", "Apply?", "Priority Score", "Overall Priority",
    "Score /100", "Stretch (1-10)", "Title", "Company", "Location", "Country", "Continent",
    "Salary", "Posted", "Source", "Duplicate Sources", "Description Available",
    "Description Quality", "Job Description", "Job Description (continued)",
    "Why It Matches", "Capability Scores", "Sponsorship risk", "Status", "AI Remarks",
    "AI Review Decision", "Apply Link", "Evidence Angle", "Honest Gaps",
    "Lead With", "Downplay", "Job Family", "Company Type", "Capability Fit",
    "Career Alignment", "Interview Probability Final", "Resume to Use", "Resume Changes Needed",
    "Resume Effort Required", "Cover Letter Effort Required", "Networking Recommendation",
    "Decision Reasoning",
]
df_final = pd.DataFrame(deduped)
if not df_final.empty:
    for col in cols:
        if col not in df_final.columns:
            df_final[col] = None
    df_final = df_final[cols]
else:
    df_final = pd.DataFrame(columns=cols)

networking_cols = [
    "Company", "Role", "Contact Name", "Contact Title", "LinkedIn URL", "Connection Sent",
    "Follow-up Due", "Response", "Referral Asked", "Notes", "Networking Recommendation", "Apply Link",
]
networking_rows = []
for r in deduped:
    if r["Bucket"] in ["A - Apply Now", "B - High Upside", "C - Network First"]:
        networking_rows.append({
            "Company": r["Company"], "Role": r["Title"],
            "Contact Name": "", "Contact Title": "", "LinkedIn URL": "",
            "Connection Sent": "No", "Follow-up Due": "", "Response": "",
            "Referral Asked": "No", "Notes": "",
            "Networking Recommendation": r.get("Networking Recommendation", ""),
            "Apply Link": r["Apply Link"],
        })
df_networking = pd.DataFrame(networking_rows, columns=networking_cols)

top20_cols = [
    "Overall Priority", "Company", "Title", "Job Family", "Evidence Angle", "Honest Gaps",
    "Lead With", "Capability Fit", "Career Alignment",
    "Interview Probability Final", "Resume to Use", "Resume Changes Needed",
    "Networking Recommendation", "Priority Score", "Apply?", "Final Bucket",
    "Decision Reasoning", "Apply Link",
]
# Sort deterministically by production priority score.
_top20_df = pd.DataFrame(deduped)
df_top20 = _top20_df.sort_values("Priority Score", ascending=False).head(20) if not _top20_df.empty else _top20_df
if not df_top20.empty:
    for col in top20_cols:
        if col not in df_top20.columns:
            df_top20[col] = None
    df_top20 = df_top20[top20_cols]
else:
    df_top20 = pd.DataFrame(columns=top20_cols)

# Employers worth a manual weekly check for this profile: their sites either
# block scraping or need their own search filters.
manual_checks = [
    # Infrastructure owners and operators
    {"Company": "National Grid", "URL": "https://careers.nationalgrid.com/", "Search": "strategy; regulation; capital planning", "Action": "Manual weekly check"},
    {"Company": "SSE", "URL": "https://www.sse.com/careers/", "Search": "strategy; investment planning; regulation", "Action": "Manual weekly check"},
    {"Company": "Octopus Energy", "URL": "https://octopus.energy/careers/", "Search": "strategy; operations; commercial", "Action": "Manual weekly check"},
    {"Company": "UK Power Networks", "URL": "https://www.ukpowernetworks.co.uk/careers", "Search": "strategy; asset management; regulation", "Action": "Manual weekly check"},
    {"Company": "Thames Water", "URL": "https://www.thameswater.co.uk/about-us/careers", "Search": "strategy; AMP8; capital planning", "Action": "Manual weekly check"},
    {"Company": "Severn Trent", "URL": "https://www.stwater.co.uk/careers/", "Search": "strategy; regulation; investment planning", "Action": "Manual weekly check"},
    {"Company": "Network Rail", "URL": "https://www.networkrail.co.uk/careers/", "Search": "strategy; planning; programme", "Action": "Manual weekly check"},
    {"Company": "Heathrow", "URL": "https://careers.heathrow.com/", "Search": "strategy; commercial; transformation", "Action": "Manual weekly check"},
    {"Company": "National Highways", "URL": "https://nationalhighways.co.uk/careers/", "Search": "strategy; investment planning", "Action": "Manual weekly check"},
    {"Company": "Transport for London", "URL": "https://tfl.gov.uk/corporate/careers/", "Search": "strategy; business planning; transformation", "Action": "Manual weekly check"},
    # Advisory and economic consulting
    {"Company": "Arup", "URL": "https://www.arup.com/careers/", "Search": "infrastructure advisory; strategy", "Action": "Referral route preferred"},
    {"Company": "Mott MacDonald", "URL": "https://www.mottmac.com/careers", "Search": "advisory; strategy; economics", "Action": "Referral route preferred"},
    {"Company": "Turner & Townsend", "URL": "https://www.turnerandtownsend.com/careers/", "Search": "infrastructure advisory; programme advisory", "Action": "Referral route preferred"},
    {"Company": "Baringa", "URL": "https://www.baringa.com/en/careers/", "Search": "energy; utilities; strategy", "Action": "Strong fit for energy strategy - referral preferred"},
    {"Company": "Frontier Economics", "URL": "https://www.frontier-economics.com/uk/en/careers/", "Search": "regulation; energy; transport", "Action": "Economics-heavy - check the entry requirements"},
    {"Company": "Oxera", "URL": "https://www.oxera.com/careers/", "Search": "economic regulation; infrastructure", "Action": "Economics-heavy - check the entry requirements"},
    # Regulators and public bodies
    {"Company": "Ofgem", "URL": "https://www.ofgem.gov.uk/careers", "Search": "price control; strategy; RIIO", "Action": "Published process - check the competency framework"},
    {"Company": "Ofwat", "URL": "https://www.ofwat.gov.uk/careers/", "Search": "price review; strategy; PR24", "Action": "Published process - check the competency framework"},
    {"Company": "NISTA (Infrastructure and Projects Authority)", "URL": "https://www.gov.uk/government/organisations/national-infrastructure-and-service-transformation-authority", "Search": "infrastructure strategy; major projects", "Action": "Apply via Civil Service Jobs"},
    {"Company": "Civil Service Jobs", "URL": "https://www.civilservicejobs.service.gov.uk/", "Search": "infrastructure strategy; economic regulation; transformation", "Action": "Best single source for public-sector infrastructure strategy"},
    # Banking and consulting, from the original list
    {"Company": "Barclays", "URL": "https://search.jobs.barclays/search-jobs", "Search": "strategy; transformation; business operations", "Action": "Manual weekly check"},
    {"Company": "HSBC", "URL": "https://mycareer.hsbc.com/en_GB/external/SearchJobs", "Search": "strategy; transformation; chief of staff", "Action": "Manual weekly check"},
    {"Company": "McKinsey", "URL": "https://www.mckinsey.com/careers/search-jobs", "Search": "associate; implementation; operations", "Action": "Referral/alumni route preferred"},
    {"Company": "BCG", "URL": "https://careers.bcg.com", "Search": "consultant; operations; transformation", "Action": "Referral/alumni route preferred"},
]

df_manual_checks = pd.DataFrame(manual_checks)

output_file = OUTPUT_FILE
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_final.to_excel(writer, index=False, sheet_name="Jobs")
    df_top20.to_excel(writer, index=False, sheet_name="Rolling Top 20")
    df_networking.to_excel(writer, index=False, sheet_name="Networking Tracker")
    df_manual_checks.to_excel(writer, index=False, sheet_name="Manual Checks")

links_made = format_workbook(output_file)
print(f"Excel formatting applied: {links_made} clickable hyperlinks across Jobs, Rolling Top 20, Networking Tracker and Manual Checks.")

full_jd_rows = sum(1 for r in deduped if (r.get("Job Description") or "").strip())
print(f"Rows carrying a job description: {full_jd_rows}/{len(deduped)} (stored in full, untrimmed).")

files.download(output_file)
print(f"Downloaded {len(deduped)} roles to {output_file} with Jobs + Rolling Top 20 + Networking Tracker + Manual Checks tabs.")


In [ ]:
# OPTIONAL MANUAL AI REVIEW STEP - OPENROUTER
# Daily production ranking above does not call AI.
# To review a shortlist, set RUN_MANUAL_AI_REVIEW = True and run this cell after the main export cell.

RUN_MANUAL_AI_REVIEW = False
AI_REVIEW_LIMIT = 40       # Use 20-40 for practical review. Set to None to attempt all rows in batches.
AI_REVIEW_BATCH_SIZE = 10  # Batch calls reduce overhead versus one request per job.

# AI configuration in one place.
MODEL_NAME = "openai/gpt-4.1-mini"
MAX_RETRIES = 2
TEMPERATURE = 0.2
MAX_TOKENS = 2500

!pip install -U openai -q

import os
import json
import re
import time
import pandas as pd
from google.colab import files
from openai import OpenAI

# Reads the OpenRouter key from Colab Secrets first, then environment variables. Never hardcode keys.
def get_openrouter_api_key():
    try:
        from google.colab import userdata
        key = userdata.get("OPENROUTER_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.environ.get("OPENROUTER_API_KEY", "")

# AI INPUT ONLY. This trimming exists to control OpenRouter token usage and is
# never applied to the Excel 'Job Description' column, which keeps the complete
# description written by the export cell.
def trim_job_description(description, max_words=1800):
    text = str(description or "").strip()
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    lower = text.lower()
    boilerplate_markers = [
        "equal opportunity", "diversity", "inclusion", "benefits", "privacy notice",
        "privacy policy", "legal notice", "about us", "about the company",
        "we are an equal", "reasonable accommodation", "background checks",
    ]
    cut_positions = [lower.find(marker) for marker in boilerplate_markers if lower.find(marker) > 400]
    if cut_positions:
        text = text[:min(cut_positions)].strip()

    words = text.split()
    if len(words) <= max_words:
        return text

    section_keywords = [
        "responsibilities", "requirements", "qualifications", "skills", "experience",
        "what you will do", "what you'll do", "about the role", "key responsibilities",
    ]
    sentences = re.split(r"(?<=[.!?])\s+", text)
    selected = []
    active = False
    for sentence in sentences:
        sentence_lower = sentence.lower()
        if any(keyword in sentence_lower for keyword in section_keywords):
            active = True
        if active:
            selected.append(sentence)
        if sum(len(s.split()) for s in selected) >= max_words:
            break

    if selected:
        return " ".join(selected).strip()
    return " ".join(words[:max_words]).strip()

# Safely extracts structured JSON arrays/objects from model responses.
def parse_ai_review_json(text):
    if not text:
        return []
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?", "", cleaned, flags=re.IGNORECASE).strip()
        cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        parsed = json.loads(cleaned)
    except Exception:
        match = re.search(r"\[.*\]|\{.*\}", cleaned, flags=re.DOTALL)
        if not match:
            return []
        try:
            parsed = json.loads(match.group(0))
        except Exception:
            return []
    if isinstance(parsed, dict):
        parsed = parsed.get("results", [])
    return parsed if isinstance(parsed, list) else []

# Builds a compact prompt payload with only fields the AI needs.
def compact_job_for_ai(row, row_id):
    return {
        "row_id": row_id,
        "job_title": row.get("Title", ""),
        "company": row.get("Company", ""),
        "location": row.get("Location", ""),
        "salary": row.get("Salary", ""),
        # Trimmed copy for the model only - row["Job Description"] is left untouched.
        "cleaned_job_description": trim_job_description(
            " ".join([row.get("Job Description", "") or "",
                      row.get("Job Description (continued)", "") or ""]).strip()
        ),
    }

# Provider abstraction so the rest of the notebook does not call OpenRouter directly.
class AIProvider:
    def __init__(self, api_key, model_name=MODEL_NAME, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
        self.client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
        self.model_name = model_name
        self.temperature = temperature
        self.max_tokens = max_tokens

    def analyze_job(self, jobs, candidate_profile):
        prompt = f"""
You are reviewing jobs for this candidate:
{candidate_profile}

For each job, judge transferable fit based only on:
Job Title, Company, Location, Salary, Cleaned Job Description.

Return ONLY valid JSON array. No markdown. No commentary.
Each item must contain:
row_id, ai_review_decision, ai_remarks

Allowed ai_review_decision values: YES, MAYBE, NETWORK, NO.
ai_remarks must be one concise sentence explaining the decision.

Jobs:
{json.dumps(jobs, ensure_ascii=False)}
"""
        last_error = None
        for attempt in range(MAX_RETRIES):
            try:
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[
                        {"role": "system", "content": "Return structured JSON only."},
                        {"role": "user", "content": prompt},
                    ],
                    temperature=self.temperature,
                    max_tokens=self.max_tokens,
                )
                content = response.choices[0].message.content
                return parse_ai_review_json(content)
            except Exception as e:
                last_error = e
                if attempt < MAX_RETRIES - 1:
                    time.sleep(2)
        print(f"OpenRouter batch failed after {MAX_RETRIES} attempt(s): {str(last_error)[:180]}")
        return []

# Built strictly from the CV. Do not add experience, seniority, salary
# expectations, visa status or achievements that are not listed here.
# Built from the Profile Bank (18 Sep 2026). Every claim below is evidenced there.
# The reviewer must not add experience, numbers, tools or qualifications.
CANDIDATE_PROFILE = """
CANDIDATE: Hemanth Dasu. Mechanical engineer who moved into Indian retail banking,
now completing a Full-Time MBA at Warwick Business School (Sep 2025 - Sep 2026).
Entering the UK job market for the first time.

CAREER, COMPLETE:
- Blue Horn, field sales, 2017, about three months. The ONLY genuine new-logo,
  cold-prospect sales experience in the whole history.
- Virtual Relationship Manager, Teleperformance for Axis Bank (2018-2019).
  Consultative sales across a 3,000+ account retail portfolio. Ranked top 10 of
  400 relationship managers in Axis Bank's annual performance competition.
- Assistant Manager, HNW Portfolio Management, HDFC Bank (2019-2021). Coordinated
  Risk, Credit and Product on complex relationships. Identified portfolio
  cross-sell as more capital-efficient than new acquisition: 18 percent revenue
  growth without expanding the client base.
- Team Leader, Relationship Management, Teleperformance for Axis Bank (2021-2024).
  Rebuilt segmentation after diagnosing a targeting gap rather than an effort gap:
  cross-sell conversion +40 percent, productivity +35 percent. Diagnosed 35 percent
  attrition as a structural development gap, not pay: attrition fell to 8 percent,
  seven associates promoted. Found centralised call routing was creating language
  and location mismatch; redesigned it: acquisition 30,000 to 40,000 a month,
  conversion +25 percent. Bancassurance across life, health and motor: 250 percent
  of motor target, 160 percent health, 120 percent life, top life performer two
  consecutive years. Designed a lead-generation approach mining the auto loan book
  for insurance prospects. Treated the renewals book as a growth channel.
- Unit Head, Commercial Operations and Acquisition, Axis Bank (2024-2025).
  Promoted at 27. Started with about 28 people on savings acquisition; by Dec 2024
  covered all seven product lines with 71 people through four Senior Team Leads.
  Built the case for 7am-10pm seven-day coverage in a video-based operation,
  including queue architecture and demand forecasting, and secured approval from
  Risk, HR and senior leadership by working each objection separately: success
  rate +40 percent, then a further +25 percent from removing onboarding
  bottlenecks. Embedded insurance at account opening: general insurance
  attachment +60 percent, life +25 percent. Renegotiated an outsourced partner's
  commercial terms and SLA scorecards: service levels 93 to 97 percent. Led a
  root-cause reframe of onboarding, misdiagnosed as a volume problem, actually
  process sequencing: customer drop-off 7 percent to 2 percent with full
  regulatory compliance. Built KPI frameworks, forecasting and demand models,
  operating rhythms and SOPs where none existed. Worked with Axis Bank's Digital
  Banking and Transformation team on customer journeys from the business side.

EDUCATION AND WORK BEYOND THE JOB:
- Warwick MBA. Dissertation on how organisations design the integration of
  AI-generated insight with human judgement for commercial decisions.
- MBA consulting project for Moasure UK: go-to-market strategy for a new product,
  TAM/SAM/SOM sizing across seven segments, financial modelling, scenario
  analysis. Recommendations adopted into commercial planning.
- Founder and President of the WBS Sports and Entertainment Club, built with no
  prior structure or budget; secured 21,000 pounds of sponsorship. Directed the
  largest-ever WBS MBAT delegation, 24 participants.
- Independent research: CERF, a four-gate framework on why technically sound
  low-carbon technologies fail to commercialise. IERF, extending it to national
  transmission infrastructure, principal case the UK's Project Union hydrogen
  backbone. A measurement framework on stakeholder-input conversion, worked
  through National Grid Electricity Transmission.
- Two self-built AI platforms: an orchestrated multi-step workflow with REST API
  integration and an LLM evaluation layer, and a conversational decision
  intelligence platform in development.
- B.Tech Mechanical Engineering. ISB Technology Entrepreneurship Programme.

TOOLS - STRICT. Safe to reference: Advanced Excel, Power BI at working-knowledge
level only, Axis Bank's internal CRM, Python automation, REST APIs, LLM
workflows, AgilePM v3 Foundation. NEVER attribute to him: SQL, Salesforce,
Dynamics or any named CRM vendor, Tableau, QuickSight, or practical Scrum Master
experience. If a role requires those, that is a genuine gap, name it.

VISA: the Graduate visa route gives two years of open work rights after the MBA.
A "we do not sponsor" advert is NOT a blocker at the point of hiring. Sponsorship
only becomes a constraint at the two-year mark. Never treat no-sponsorship
wording as a reason to score a role down.

WHERE HE HAS A REAL SHOT: commercial strategy, commercial operations and
commercial excellence; business development and partnerships built on growing an
existing base rather than cold hunting; transformation, operating-model and
process-redesign work, especially in regulated industries; builder roles that
require creating structure from nothing, including enablement, GTM operations and
programme management; AI-adjacent operations or strategy where the requirement is
understanding AI capability and its limits rather than ML engineering; and
infrastructure or utilities strategy and policy roles, where CERF and IERF give a
real citable point of engagement.

GENUINE STRETCHES, NAME THE GAP RATHER THAN HIDING IT: new-business-heavy B2B
sales where cold prospecting is the core motion, since only the three-month Blue
Horn stint is genuine new-logo evidence; enterprise or government sales motions;
industrial and engineering strategy consulting, where the engineering identity is
real but has not been the day job for seven years; and insurance roles at
commercial-lines employers, since all his insurance experience is personal lines.

DO NOT RECOMMEND APPLYING: marketing management roles, since no marketing-function
ownership exists anywhere; professional-services or legal business development,
where pitch/RFP and sector BD are hard essential gates; deep quantitative or
specialist finance; pure AI or ML engineering roles; and entry-level
individual-contributor banking roles with no strategic scope.

HOW TO WRITE THE REVIEW: be direct and plain. Name a genuine gap once, clearly,
and follow it with what actually transfers. Never invent a number or a tool.
Never claim equivalence that would unravel under one follow-up question. He has
stated he prefers an honest limitation over confident overreach, and that this
preference overrides any temptation to inflate a match score.
"""

if not RUN_MANUAL_AI_REVIEW:
    print("Manual AI review is OFF. Set RUN_MANUAL_AI_REVIEW = True, add OPENROUTER_API_KEY in Colab Secrets, then run this cell to fill AI Remarks.")
else:
    api_key = get_openrouter_api_key()
    if not api_key:
        print("Missing OPENROUTER_API_KEY. Add it in Colab Secrets before running manual AI review.")
    else:
        provider = AIProvider(api_key=api_key)

        review_rows = list(deduped)
        review_rows = sorted(review_rows, key=lambda r: r.get("Priority Score", 0) or 0, reverse=True)
        if AI_REVIEW_LIMIT is not None:
            review_rows = review_rows[:AI_REVIEW_LIMIT]

        row_lookup = {}
        compact_rows = []
        for row_id, row in enumerate(review_rows, 1):
            row_lookup[row_id] = row
            compact_rows.append(compact_job_for_ai(row, row_id))

        print(f"Manual AI review via OpenRouter: {len(compact_rows)} jobs in batches of {AI_REVIEW_BATCH_SIZE}. Model: {MODEL_NAME}")
        for start in range(0, len(compact_rows), AI_REVIEW_BATCH_SIZE):
            batch = compact_rows[start:start + AI_REVIEW_BATCH_SIZE]
            reviews = provider.analyze_job(batch, CANDIDATE_PROFILE)
            for review in reviews:
                row = row_lookup.get(int(review.get("row_id", 0) or 0))
                if not row:
                    continue
                row["AI Remarks"] = review.get("ai_remarks", "")
                row["AI Review Decision"] = review.get("ai_review_decision", "")
            time.sleep(1)

        df_ai_jobs = pd.DataFrame(deduped)
        for col in cols:
            if col not in df_ai_jobs.columns:
                df_ai_jobs[col] = None
        df_ai_jobs = df_ai_jobs[cols]

        df_ai_top20 = df_ai_jobs.sort_values("Priority Score", ascending=False).head(20)
        df_ai_top20 = df_ai_top20[[c for c in top20_cols if c in df_ai_top20.columns]]
        df_ai_networking = pd.DataFrame(networking_rows, columns=networking_cols)
        df_ai_manual_checks = df_manual_checks.copy()

        output_file = AI_OUTPUT_FILE
        with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
            df_ai_jobs.to_excel(writer, index=False, sheet_name="Jobs")
            df_ai_top20.to_excel(writer, index=False, sheet_name="Rolling Top 20")
            df_ai_networking.to_excel(writer, index=False, sheet_name="Networking Tracker")
            df_ai_manual_checks.to_excel(writer, index=False, sheet_name="Manual Checks")

        ai_links = format_workbook(output_file)
        print(f"Excel formatting applied: {ai_links} clickable hyperlinks.")
        files.download(output_file)
        print(f"Downloaded AI-reviewed workbook: {output_file}")


In [ ]:
ubs_found = [r for r in deduped if "ubs" in r["Company"].lower()]
print(f"UBS roles in current results: {len(ubs_found)}")
for r in ubs_found:
    print(f"  [{r['Score /100']}] {r['Title']} @ {r['Company']} â€” {r['Location']} | Source: {r['Source']}")

In [ ]:
# Check if UBS came through LinkedIn before scoring
ubs_raw = [j for j in linkedin_jobs if "ubs" in j.get("company", "").lower()]
print(f"UBS in LinkedIn raw: {len(ubs_raw)}")
for j in ubs_raw:
    print(f"  {j['title']} â€” {j['location']}")

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re

headers_ubs = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-GB,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Referer": "https://jobs.ubs.com",
}

UBS_URLS = [
    "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=searchResults&SearchType=linkquery&LinkID=15231",
    "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=JobListing&noback=1",
    "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=searchResults&SearchType=linkquery&LinkID=15231&keyWordSearch=strategy+operations&locationSearch=London",
    "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=searchResults&SearchType=linkquery&LinkID=15231&keyWordSearch=transformation&locationSearch=London",
    "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=searchResults&SearchType=linkquery&LinkID=15231&keyWordSearch=infrastructure+strategy&locationSearch=London",
]

# Also try their API endpoint
UBS_API_URLS = [
    "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=searchResults&SearchType=linkquery&LinkID=15231&keyWordSearch=strategy+operations+operations&locationSearch=London&format=json",
    "https://jobs.ubs.com/talentcommunity/api/v1/jobs?partnerid=25008&siteid=5012&keywords=strategy+operations&location=London",
    "https://jobs.ubs.com/api/jobs?keywords=strategy+operations&location=London&partnerid=25008",
]

RELEVANT = RELEVANT_TITLES  # candidate taxonomy from the settings cell
EXCLUDE = HARD_EXCLUDE      # context-aware filtering via title_is_excluded()

ubs_jobs = []

print("Testing UBS TGnewUI platform...")
print("-" * 55)

session = requests.Session()

for url in UBS_URLS:
    try:
        r = session.get(url, headers=headers_ubs, timeout=15)
        print(f"  {url[-80:]}")
        print(f"    Status: {r.status_code} | Size: {len(r.text)} chars")

        if r.status_code == 200 and len(r.text) > 1000:
            soup = BeautifulSoup(r.text, "html.parser")

            # Look for job listings in various formats
            found = 0

            # Method 1: Find job title links
            for a in soup.find_all("a", href=True):
                title = a.get_text(strip=True)
                href = a["href"]
                if len(title) < 8 or len(title) > 110: continue
                if not any(w in title.lower() for w in RELEVANT): continue
                if title_is_excluded(title): continue
                full_url = href if href.startswith("http") else f"https://jobs.ubs.com{href}"
                if any(j["url"] == full_url for j in ubs_jobs): continue
                ubs_jobs.append({
                    "title": title, "company": "UBS",
                    "location": "London, UK", "salary": "See listing",
                    "posted": "Live now", "desc": title,
                    "source": "Career Page", "url": full_url,
                })
                found += 1
                print(f"    âœ… Found: {title}")

            # Method 2: Look for JSON data in page
            scripts = soup.find_all("script")
            for script in scripts:
                if script.string and "jobTitle" in str(script.string):
                    print(f"    JSON data found in script tag â€” {len(script.string)} chars")
                    # Try to extract job titles
                    titles = re.findall(r'"jobTitle"\s*:\s*"([^"]+)"', script.string)
                    for title in titles:
                        if not any(w in title.lower() for w in RELEVANT): continue
                        if title_is_excluded(title): continue
                        print(f"    âœ… JSON job: {title}")

            # Method 3: Extract all text that looks like job titles
            all_text = soup.get_text(separator="\n")
            lines = [l.strip() for l in all_text.split("\n") if l.strip()]
            text_found = 0
            for line in lines:
                if len(line) < 10 or len(line) > 100: continue
                if not any(w in line.lower() for w in RELEVANT): continue
                if any(w in line.lower() for w in EXCLUDE): continue
                if any(nav in line.lower() for nav in ["cookie", "privacy", "sign in",
                    "log in", "search", "filter", "sort", "home", "about",
                    "contact", "careers at", "why ubs", "our culture"]): continue
                if text_found < 20:
                    print(f"    Text: {line}")
                text_found += 1

    except Exception as e:
        print(f"  Error: {e}")
    time.sleep(1)

# Try API endpoints
print()
print("Testing UBS API endpoints...")
print("-" * 55)
for url in UBS_API_URLS:
    try:
        r = session.get(url, headers={**headers_ubs, "Accept": "application/json"}, timeout=12)
        print(f"  Status: {r.status_code} | Size: {len(r.text)} | URL: {url[-60:]}")
        if r.status_code == 200:
            print(f"  Preview: {r.text[:300]}")
    except Exception as e:
        print(f"  Error: {e}")

print()
print(f"UBS jobs found: {len(ubs_jobs)}")

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import json

headers_ubs = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-GB,en;q=0.9",
}

session = requests.Session()
url = "https://jobs.ubs.com/TGnewUI/Search/home/HomeWithPreLoad?partnerid=25008&siteid=5012&PageType=searchResults&SearchType=linkquery&LinkID=15231"

r = session.get(url, headers=headers_ubs, timeout=20)
soup = BeautifulSoup(r.text, "html.parser")

print(f"Page size: {len(r.text)} chars")
print()

# Extract all script tags and look for job data
scripts = soup.find_all("script")
print(f"Total script tags: {len(scripts)}")

for i, script in enumerate(scripts):
    if not script.string:
        continue
    s = script.string

    # Look for job-related JSON keys
    if any(key in s for key in ["jobTitle", "JobTitle", "job_title", "position",
                                  "requisition", "Requisition", "openings"]):
        print(f"\n=== Script {i} ({len(s)} chars) ===")
        print(f"Preview: {s[:500]}")
        print()

        # Try to find all job title patterns
        patterns = [
            r'"jobTitle"\s*:\s*"([^"]+)"',
            r'"JobTitle"\s*:\s*"([^"]+)"',
            r'"title"\s*:\s*"([^"]+)"',
            r'"Title"\s*:\s*"([^"]+)"',
            r'"name"\s*:\s*"([^"]+)"',
            r'"position"\s*:\s*"([^"]+)"',
            r'"RequisitionTitle"\s*:\s*"([^"]+)"',
        ]

        for pattern in patterns:
            matches = re.findall(pattern, s)
            if matches:
                print(f"Pattern '{pattern[:30]}' found {len(matches)} matches:")
                for m in matches[:10]:
                    print(f"  - {m}")

        # Try to parse as JSON
        json_matches = re.findall(r'\{[^{}]{100,}\}', s)
        for jm in json_matches[:3]:
            try:
                data = json.loads(jm)
                print(f"Valid JSON object found with keys: {list(data.keys())[:10]}")
            except:
                pass